# NEHM: **LongCLIP-B** (ViT-B/16) + student — **Pipeline B v3.1 (16+1 scale-adaptive multiview)** (Colab CUDA)

This notebook trains Pipeline B against a **2304-D fused teacher** and writes B-suffixed checkpoints.

**B v3.1 contract (what actually runs).**

- Backbone: **LongCLIP-B** (ViT-B/16 visual, 248-token text tower).
- Image embeddings loaded from `output_random_local_global_B/image_embeddings/` must be **768-D** (`ln_post` pre-proj).
- Text embeddings loaded from `output_random_local_global_B/column_text_embeddings/` must be **512-D**.
- Fused teacher = `[image_768 ⨁ illum_512 ⨁ avg(class+subclass)_512 ⨁ avg(desc+meta)_512]` = **2304-D**.
- Student head: `_IMG_D -> 1024 -> 1536 -> edim`, where `_IMG_D` comes from image npys and `edim` is the fused teacher dim.

**View layout (teacher and student must match).**

- 1 global 224x224 view (`short_side` resize + center crop, with 1-pixel border trim).
- 16 random **scale-adaptive native tiles** per image:

  `tile_px = clip(round(TILE_MAX_PX * sqrt(W*H) / TILE_SCALE_REF_PX), TILE_MIN_PX, TILE_MAX_PX)`

- Set `TILE_PX > 0` to run fixed-size-tile ablations; default `TILE_PX = 0` is B v3.1.

**Training order.**

- `Z_embed` warmup on image embedding -> fused teacher regression.
- ViT progressive stages: `A_readout -> B_last2 -> C_last4 -> C_last8 -> C_last12 -> D_full_decay`.
- Global-best checkpoint: `vit_student_finetune_best_B_logitb.pth` by `val_acc_labeled`.

**Critical safety behavior.**

- The notebook validates image/text dims before training.
- The notebook writes run config + summary manifests to keep each checkpoint traceable.
- If a pre-existing best checkpoint has incompatible teacher dim, the notebook can be configured to fail fast instead of silently mixing contracts.

**Required Drive artifacts.**

- `nehm_pipeline/`, `Database/Material_Database.xlsx`, `NEHM/`, `LongCLIP/checkpoints/longclip-B.pt`.
- B v3.1 teacher embeddings from `Notebooks/Teacher_inference_embeddings_B.ipynb`:
  - `output_random_local_global_B/image_embeddings/image_embeddings_B_random_pooled.npy`
  - `output_random_local_global_B/image_embeddings/image_embeddings_B_global.npy`
  - `output_random_local_global_B/column_text_embeddings/columns_manifest.csv` + column npys.


In [ ]:
# LongCLIP + Excel / sklearn (Colab ships PyTorch). LongCLIP itself is a vendored
# package under PROJECT_ROOT/LongCLIP/, so we only install its tokenizer / text deps here.
!pip install -q pandas openpyxl scikit-learn tqdm ftfy regex


In [ ]:
from __future__ import annotations
import sys
from pathlib import Path

# --- EDIT: folder under My Drive containing nehm_pipeline/, Database/, NEHM/, NEHM_RESULTS/ ---
DRIVE_FOLDER = "NEHM_Project"

from google.colab import drive

drive.mount("/content/drive", force_remount=False)

PROJECT_ROOT = (Path("/content/drive/MyDrive") / DRIVE_FOLDER).resolve()
if not PROJECT_ROOT.is_dir():
    raise FileNotFoundError(
        f"Expected project at {PROJECT_ROOT}\n"
        "Fix DRIVE_FOLDER or sync your repo — see colab/MIGRATE_TO_DRIVE.md."
    )
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from colab_env import assert_training_artifacts

assert_training_artifacts(PROJECT_ROOT, pipeline="B")
print("PROJECT_ROOT =", PROJECT_ROOT)


In [ ]:
# Tunable parameters (edit this cell only)
# ------------------------------------------------
PIPELINE_OUT_REL = 'NEHM_RESULTS/pipeline_notebook'
IMAGES_DIR_REL = 'NEHM'
MATERIAL_DB_REL = 'Database/Material_Database.xlsx'
STUDENT_CKPT_REL = 'NEHM_RESULTS/student_clip_student.pth'

EMB_ROOT_B_REL = 'LongCLIP_Embeddings_v1/output_random_local_global_B'
IMAGE_WEIGHT_RANDOM = 0.5
IMAGE_WEIGHT_GLOBAL = 0.5

LONGCLIP_ROOT_REL = 'LongCLIP'
LONGCLIP_CHECKPOINT_REL = 'LongCLIP/checkpoints/longclip-B.pt'
CLIP_MODEL = 'LongCLIP-B'
EXPECTED_IMAGE_DIM = 768
EXPECTED_TEXT_EMBED_DIM = 512
EXPECTED_TEACHER_DIM = EXPECTED_IMAGE_DIM + 3 * EXPECTED_TEXT_EMBED_DIM
DEVICE = 'cuda'
USE_MIXED_PRECISION = True
USE_BF16_AUTOCAST = True

BATCH_SIZE = 96
BATCH_SIZE_VIT_SUBSEQUENT_MAX = 16
BATCH_SIZE_STAGE_C = 8
BATCH_SIZE_STAGE_D = 8
DATALOADER_NUM_WORKERS = 0
DATALOADER_PREFETCH_FACTOR = 4

N_NATIVE_RANDOM = 16
TILE_PX = 0
TILE_MIN_PX = 16
TILE_MAX_PX = 224
TILE_SCALE_REF_PX = 3000
CROP_SIZE = 224
GLOBAL_KIND = 'short_side'
PRE_CROP_MARGIN_PX = 1
WINDOW_SAMPLE_SEED = 42
VAL_FRACTION = 0.15
SPLIT_SEED = 42

MATERIAL_LOGIT_COLUMN = 'logitb'
NUM_MATERIAL_CLASSES = 513
EMB_LOSS_KIND = 'l1'
LAMBDA_EMB_P3 = 0.80
LAMBDA_CE_P3 = 0.20
LABEL_SMOOTHING_P3 = 0.10

RUN_PROGRESSIVE_VIT_FINETUNE = True
PREFER_SAVE_CHECKPOINTS_TO_GOOGLE_DRIVE = True
DRIVE_PROJECT_FOLDER = 'NEHM_Project'
ALLOW_INCOMPATIBLE_BEST_CHECKPOINT = False
FINETUNE_CKPT_BEST_NAME = 'vit_student_finetune_best_B_logitb.pth'
TRAINING_LOG_JSONL_NAME = 'vit_finetune_training_history_B.jsonl'
RESUME_VIT_PRIORITY_FILENAMES = [
    'vit_student_finetune_best_B_logitb.pth',
    'vit_student_finetune_best_B.pth',
    'vit_finetune_stage_B_D_full_decay.pth',
    'vit_finetune_stage_B_C_last12.pth',
    'vit_finetune_stage_B_C_last8.pth',
    'vit_finetune_stage_B_C_last4.pth',
    'vit_finetune_stage_B_B_last2.pth',
    'vit_finetune_stage_B_A_readout.pth',
    'vit_finetune_stage_B_Z_embed_student_head.pth',
]

USE_CLASS_BALANCED_SAMPLER = True
USE_CE_CLASS_WEIGHTS = False
SAMPLER_WEIGHT_KIND = 'inv_sqrt'

STUDENT_INPUT_DROPOUT = 0.05
STUDENT_DECODER_DROPOUT = 0.0
STUDENT_CLASSIFIER_DROPOUT = 0.2
TEACHER_TARGET_NOISE_STD = 0.005

USE_SWA_IN_STAGE_D = True
SWA_LAST_N_EPOCHS = 3
A_VIT_PROMOTE_MIN_VS_Z = False
STAGE_C_USE_FP32 = False
STAGE_C_VISUAL_ENCODE_CHUNK_SIZE = 16
STAGE_D_USE_FP32 = False

MAX_BACKUP_KEEP = 5
USE_CHANNELS_LAST = True
USE_TORCH_COMPILE_HEAD = False
MIRROR_IMAGES_DIR = False

START_FROM_BACKBONE_ONLY = False
SCHEDULE_START_STAGE = None

GEOM_AUGMENT_TRAIN = True
AUGMENT_MAX_ROTATION_DEG = 5.0
AUGMENT_HFLIP_P = 0.5
AUGMENT_VFLIP_P = 0.5

GRAD_CLIP_FINETUNE = 1.0
WEIGHT_DECAY_FINETUNE = 0.05
LAYER_DECAY_FULL = 0.75
SHOW_TQDM_BATCHES = True
LOG_EVERY_N_TRAIN_BATCHES = 0
LOG_EVERY_N_VAL_BATCHES = 0

EPOCHS_VIT_A = 4
EPOCHS_VIT_A0 = 4
LR_READOUT_A0 = 5e-5
A_VIT_MAX_EPOCHS_PER_STAGE = 9
A_VIT_MIN_EPOCHS_PER_STAGE = 4
A_VIT_EARLY_STOP_PATIENCE = 8
EARLY_STOP_PATIENCE_EMBED = 25
Z_EMBED_TARGET_VAL_ACC = None
RUN_EMBEDDING_STUDENT_WARMUP = True
EPOCHS_EMBED_WARMUP = 50
LR_EMBED_WARMUP = 5e-4
EMBED_WARMUP_BATCH_SIZE = 512
EMBED_WARMUP_WEIGHT_DECAY = 0.05
EMBED_WARMUP_GRAD_CLIP = 1.0
EPOCHS_VIT_B = 8
EPOCHS_VIT_C4 = 6
EPOCHS_VIT_C8 = 8
EPOCHS_VIT_C12 = 10
EPOCHS_VIT_D = 12

EARLY_STOP_PATIENCE_VIT = {
    'B_last2': 6,
    'C_last4': 6,
    'C_last8': 8,
    'C_last12': 10,
    'D_full_decay': 12,
}

LR_STUDENT_A, LR_READOUT_A = 1e-4, 5e-5
LR_STUDENT_B, LR_READOUT_B, LR_BACKBONE_B = 8e-5, 3e-5, 1.5e-5
LR_STUDENT_C, LR_READOUT_C = 6e-5, 2e-5
LR_BACKBONE_C4, LR_BACKBONE_C8, LR_BACKBONE_C12 = 1.2e-5, 8e-6, 5e-6
LR_STUDENT_D, LR_READOUT_D, LR_BACKBONE_LAST_D = 5e-5, 2e-5, 4e-6

print('Tunable parameters loaded.')

In [ ]:
import importlib
import torch
from torch.utils.data import DataLoader

import nehm_pipeline.preprocess_image as preimg
import nehm_pipeline.clip_vit_multiview_finetune as vfin
import nehm_pipeline.clip_visual_student as cvs
import nehm_pipeline.student_clip as sc

importlib.reload(preimg)
importlib.reload(vfin)
importlib.reload(cvs)
importlib.reload(sc)

from nehm_pipeline.clip_vit_multiview_finetune import (
    LabeledMultiviewFinetuneDataset,
    build_adamw_vit_finetune,
    collate_multiview_pil_teacher,
    freeze_clip_visual,
    unfreeze_clip_visual_ln_post_proj_only,
    unfreeze_clip_visual_last_n_blocks,
    unfreeze_clip_visual_full,
    vit_finetune_run_epoch,
    vit_finetune_unfreeze_schedule_doc,
)
from nehm_pipeline.device import (
    clip_visual_preprocess_without_channel_normalize,
    resolve_device,
    dataloader_pin_memory,
    apply_pytorch_training_perf_for_vit_student,
)
from nehm_pipeline.student_clip import (
    ImageEmbeddingStudent,
    load_distillation_bundle,
    load_material_logits_from_excel,
    train_val_split_rows,
)


def make_cuda_grad_scaler():
    """GradScaler for AMP (defined here so Colab works if Drive lacks an updated clip_vit_multiview_finetune)."""
    try:
        return torch.amp.GradScaler("cuda")
    except (AttributeError, TypeError):
        return torch.cuda.amp.GradScaler()


print(vit_finetune_unfreeze_schedule_doc())


In [ ]:
import os
import json as _json
import random as _random
import shutil as _shutil
import time as _time
from datetime import datetime as _dt, timezone as _tz

# Long Pro+ runs: less CUDA allocator fragmentation (set before first GPU alloc).
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import numpy as np
import torch

try:
    import google.colab  # noqa: F401

    _ON_COLAB = True
except ImportError:
    _ON_COLAB = False


GLOBAL_SEED = 42


def seed_everything(seed: int = GLOBAL_SEED) -> None:
    """Reproducibility: seed Python, NumPy, Torch CPU/CUDA. cuDNN benchmark stays on for speed."""
    _random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


seed_everything(GLOBAL_SEED)


LOCAL_CACHE_DIR = Path("/content/cache_nehm") if _ON_COLAB else (PROJECT_ROOT / "_local_cache")
LOCAL_CACHE_DIR.mkdir(parents=True, exist_ok=True)


def cache_drive_file(src: Path, *, refresh: bool = False) -> Path:
    """Copy a Drive file to fast local SSD on first hit; return the local path."""
    src = Path(src)
    if not src.is_file():
        raise FileNotFoundError(src)
    if not _ON_COLAB:
        return src
    dst = LOCAL_CACHE_DIR / src.name
    if dst.is_file() and not refresh and dst.stat().st_size == src.stat().st_size:
        return dst
    print(f"  caching {src.name} → {dst} ({src.stat().st_size / 2**20:.1f} MiB)", flush=True)
    _shutil.copy2(src, dst)
    return dst


def cache_drive_dir(src: Path, *, refresh: bool = False, name: str | None = None) -> Path:
    """Mirror a Drive directory to local SSD via ``cp -a`` (single subprocess; far
    faster than ``shutil.copytree`` on Drive FUSE which round-trips per-file).
    Idempotent. Skipped off-Colab.
    """
    import subprocess
    src = Path(src)
    if not src.is_dir():
        raise FileNotFoundError(src)
    if not _ON_COLAB:
        return src
    dst = LOCAL_CACHE_DIR / (name or src.name)
    if dst.is_dir() and not refresh:
        return dst
    print(f"  mirroring {src} → {dst} (subprocess cp -a; this can take a while for many small files)...", flush=True)
    t0 = _time.time()
    if dst.exists():
        _shutil.rmtree(dst)
    dst.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(["cp", "-a", str(src), str(dst)], check=True)
    print(f"    done ({_time.time() - t0:.1f}s, {sum(1 for _ in dst.rglob('*'))} files)", flush=True)
    return dst


def atomic_torch_save(obj, path: Path) -> Path:
    """torch.save with tmp file + atomic rename, so partial writes never leave a corrupt file on Drive."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    torch.save(obj, tmp)
    os.replace(tmp, path)
    return path


def prune_old_backups(directory: Path, *, stem_prefix: str, keep: int = 5) -> int:
    """Keep at most ``keep`` newest ``<stem_prefix>.bak_*`` files in ``directory``; delete the rest."""
    if keep <= 0:
        return 0
    directory = Path(directory)
    if not directory.is_dir():
        return 0
    bak = sorted(
        (p for p in directory.iterdir() if p.is_file() and p.name.startswith(f"{stem_prefix}.bak_")),
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )
    removed = 0
    for p in bak[keep:]:
        try:
            p.unlink()
            removed += 1
        except OSError:
            pass
    return removed

from nehm_pipeline.clip_visual_student import (
    ClipVisualStudent,
    load_clip_visual_student_checkpoint_skip_classifier,
)

# Strict pass guard: ensure every tunable key exists even if cell 3 was partially edited.
# Existing user values are preserved; only missing keys receive defaults.
from copy import deepcopy as _deepcopy

_TUNABLE_DEFAULTS = {
    'PIPELINE_OUT_REL': 'NEHM_RESULTS/pipeline_notebook',
    'IMAGES_DIR_REL': 'NEHM',
    'MATERIAL_DB_REL': 'Database/Material_Database.xlsx',
    'STUDENT_CKPT_REL': 'NEHM_RESULTS/student_clip_student.pth',
    'EMB_ROOT_B_REL': 'LongCLIP_Embeddings_v1/output_random_local_global_B',
    'IMAGE_WEIGHT_RANDOM': 0.5,
    'IMAGE_WEIGHT_GLOBAL': 0.5,
    'LONGCLIP_ROOT_REL': 'LongCLIP',
    'LONGCLIP_CHECKPOINT_REL': 'LongCLIP/checkpoints/longclip-B.pt',
    'CLIP_MODEL': 'LongCLIP-B',
    'EXPECTED_IMAGE_DIM': 768,
    'EXPECTED_TEXT_EMBED_DIM': 512,
    'EXPECTED_TEACHER_DIM': 768 + 3 * 512,
    'DEVICE': 'cuda',
    'USE_MIXED_PRECISION': True,
    'USE_BF16_AUTOCAST': True,
    'BATCH_SIZE': 96,
    'BATCH_SIZE_VIT_SUBSEQUENT_MAX': 16,
    'BATCH_SIZE_STAGE_C': 8,
    'BATCH_SIZE_STAGE_D': 8,
    'DATALOADER_NUM_WORKERS': 0,
    'DATALOADER_PREFETCH_FACTOR': 4,
    'N_NATIVE_RANDOM': 16,
    'TILE_PX': 0,
    'TILE_MIN_PX': 16,
    'TILE_MAX_PX': 224,
    'TILE_SCALE_REF_PX': 3000,
    'CROP_SIZE': 224,
    'GLOBAL_KIND': 'short_side',
    'PRE_CROP_MARGIN_PX': 1,
    'WINDOW_SAMPLE_SEED': 42,
    'VAL_FRACTION': 0.15,
    'SPLIT_SEED': 42,
    'MATERIAL_LOGIT_COLUMN': 'logitb',
    'NUM_MATERIAL_CLASSES': 308,
    'EMB_LOSS_KIND': 'l1',
    'LAMBDA_EMB_P3': 1.0,
    'LAMBDA_CE_P3': 0.3,
    'LABEL_SMOOTHING_P3': 0.10,
    'RUN_PROGRESSIVE_VIT_FINETUNE': True,
    'PREFER_SAVE_CHECKPOINTS_TO_GOOGLE_DRIVE': True,
    'DRIVE_PROJECT_FOLDER': 'NEHM_Project',
    'ALLOW_INCOMPATIBLE_BEST_CHECKPOINT': False,
    'FINETUNE_CKPT_BEST_NAME': 'vit_student_finetune_best_B_logitb.pth',
    'TRAINING_LOG_JSONL_NAME': 'vit_finetune_training_history_B.jsonl',
    'RESUME_VIT_PRIORITY_FILENAMES': [
        'vit_student_finetune_best_B_logitb.pth',
        'vit_student_finetune_best_B.pth',
        'vit_finetune_stage_B_D_full_decay.pth',
        'vit_finetune_stage_B_C_last12.pth',
        'vit_finetune_stage_B_C_last8.pth',
        'vit_finetune_stage_B_C_last4.pth',
        'vit_finetune_stage_B_B_last2.pth',
        'vit_finetune_stage_B_A_readout.pth',
        'vit_finetune_stage_B_Z_embed_student_head.pth',
    ],
    'USE_CLASS_BALANCED_SAMPLER': True,
    'USE_CE_CLASS_WEIGHTS': False,
    'SAMPLER_WEIGHT_KIND': 'inv_sqrt',
    'STUDENT_INPUT_DROPOUT': 0.05,
    'STUDENT_DECODER_DROPOUT': 0.0,
    'STUDENT_CLASSIFIER_DROPOUT': 0.2,
    'TEACHER_TARGET_NOISE_STD': 0.005,
    'USE_SWA_IN_STAGE_D': True,
    'SWA_LAST_N_EPOCHS': 3,
    'A_VIT_PROMOTE_MIN_VS_Z': False,
    'STAGE_C_USE_FP32': False,
    'STAGE_C_VISUAL_ENCODE_CHUNK_SIZE': 16,
    'STAGE_D_USE_FP32': False,
    'MAX_BACKUP_KEEP': 5,
    'USE_CHANNELS_LAST': True,
    'USE_TORCH_COMPILE_HEAD': False,
    'MIRROR_IMAGES_DIR': False,
    'START_FROM_BACKBONE_ONLY': False,
    'SCHEDULE_START_STAGE': None,
    'GEOM_AUGMENT_TRAIN': True,
    'AUGMENT_MAX_ROTATION_DEG': 5.0,
    'AUGMENT_HFLIP_P': 0.5,
    'AUGMENT_VFLIP_P': 0.5,
    'GRAD_CLIP_FINETUNE': 1.0,
    'WEIGHT_DECAY_FINETUNE': 0.05,
    'LAYER_DECAY_FULL': 0.75,
    'SHOW_TQDM_BATCHES': True,
    'LOG_EVERY_N_TRAIN_BATCHES': 0,
    'LOG_EVERY_N_VAL_BATCHES': 0,
    'EPOCHS_VIT_A': 4,
    'EPOCHS_VIT_A0': 4,
    'LR_READOUT_A0': 5e-5,
    'A_VIT_MAX_EPOCHS_PER_STAGE': 9,
    'A_VIT_MIN_EPOCHS_PER_STAGE': 4,
    'A_VIT_EARLY_STOP_PATIENCE': 8,
    'EARLY_STOP_PATIENCE_EMBED': 25,
    'Z_EMBED_TARGET_VAL_ACC': None,
    'RUN_EMBEDDING_STUDENT_WARMUP': True,
    'EPOCHS_EMBED_WARMUP': 50,
    'LR_EMBED_WARMUP': 5e-4,
    'EMBED_WARMUP_BATCH_SIZE': 512,
    'EMBED_WARMUP_WEIGHT_DECAY': 0.05,
    'EMBED_WARMUP_GRAD_CLIP': 1.0,
    'EPOCHS_VIT_B': 8,
    'EPOCHS_VIT_C4': 6,
    'EPOCHS_VIT_C8': 8,
    'EPOCHS_VIT_C12': 10,
    'EPOCHS_VIT_D': 12,
    'EARLY_STOP_PATIENCE_VIT': {
        'B_last2': 6,
        'C_last4': 6,
        'C_last8': 8,
        'C_last12': 10,
        'D_full_decay': 12,
    },
    'LR_STUDENT_A': 1e-4,
    'LR_READOUT_A': 5e-5,
    'LR_STUDENT_B': 8e-5,
    'LR_READOUT_B': 3e-5,
    'LR_BACKBONE_B': 1.5e-5,
    'LR_STUDENT_C': 6e-5,
    'LR_READOUT_C': 2e-5,
    'LR_BACKBONE_C4': 1.2e-5,
    'LR_BACKBONE_C8': 8e-6,
    'LR_BACKBONE_C12': 5e-6,
    'LR_STUDENT_D': 5e-5,
    'LR_READOUT_D': 2e-5,
    'LR_BACKBONE_LAST_D': 4e-6,
}
for _k, _v in _TUNABLE_DEFAULTS.items():
    if _k not in globals():
        globals()[_k] = _deepcopy(_v)

PIPELINE_OUT = PROJECT_ROOT / PIPELINE_OUT_REL
IMAGES_DIR = PROJECT_ROOT / IMAGES_DIR_REL
MATERIAL_DB_XLSX = PROJECT_ROOT / MATERIAL_DB_REL
STUDENT_CKPT = PROJECT_ROOT / STUDENT_CKPT_REL

# Pipeline B v3 teacher: image (random + global) + text npys produced by
# Notebooks/Teacher_inference_embeddings_B.ipynb under the LongCLIP-B backbone.
EMB_ROOT_B = PROJECT_ROOT / EMB_ROOT_B_REL
IMAGE_EMB_ROOT_B = EMB_ROOT_B / "image_embeddings"
IMAGE_EMB_GLOBAL = IMAGE_EMB_ROOT_B / "image_embeddings_B_global.npy"
IMAGE_EMB_RANDOM = IMAGE_EMB_ROOT_B / "image_embeddings_B_random_pooled.npy"
TEXT_COLUMNS_MANIFEST = EMB_ROOT_B / "column_text_embeddings" / "columns_manifest.csv"
# Pipeline A's exact image-teacher recipe: equal weighting on the L2-normalized random pool
# and global view, then re-L2 the blend. Lower IMAGE_WEIGHT_RANDOM if your dataset has tiny
# native short-sides where tiles add only blur (not the case for B's 5,400-row dataset).
IMAGE_WEIGHT_RANDOM = float(IMAGE_WEIGHT_RANDOM)
IMAGE_WEIGHT_GLOBAL = float(IMAGE_WEIGHT_GLOBAL)

# Backbone — LongCLIP-B (ViT-B/16, 512-D image + 248-token text). The LongCLIP package itself
# lives at PROJECT_ROOT/LongCLIP/ as a vendored relative-import package; the loader function
# below registers it on sys.path correctly. The checkpoint must match the dimensionality of
# the saved npys (LongCLIP-B → 512-D; LongCLIP-L → 768-D). Switching variants requires
# regenerating both image AND text npys to avoid a silent shape mismatch.
_longclip_root_requested = PROJECT_ROOT / LONGCLIP_ROOT_REL
_longclip_root_candidates = [
    _longclip_root_requested,
    PROJECT_ROOT / 'NEHM_Stable' / LONGCLIP_ROOT_REL,
    PROJECT_ROOT.parent / 'NEHM_Stable' / LONGCLIP_ROOT_REL,
]
LONGCLIP_ROOT = next((p for p in _longclip_root_candidates if p.is_dir()), _longclip_root_requested)

_longclip_ckpt_requested = PROJECT_ROOT / LONGCLIP_CHECKPOINT_REL
_longclip_ckpt_candidates = [
    _longclip_ckpt_requested,
    LONGCLIP_ROOT / 'checkpoints' / Path(LONGCLIP_CHECKPOINT_REL).name,
    PROJECT_ROOT / 'NEHM_Stable' / 'LongCLIP' / 'checkpoints' / Path(LONGCLIP_CHECKPOINT_REL).name,
]
LONGCLIP_CHECKPOINT = next((p for p in _longclip_ckpt_candidates if p.is_file()), _longclip_ckpt_requested)
CLIP_MODEL = str(CLIP_MODEL)  # metadata label
EXPECTED_IMAGE_DIM = int(EXPECTED_IMAGE_DIM)       # LongCLIP-B ViT-B/16: pooled ln_post CLS (no visual.proj)
EXPECTED_TEXT_EMBED_DIM = int(EXPECTED_TEXT_EMBED_DIM)  # text EOT after ln_final (pre text_projection)
EXPECTED_TEACHER_DIM = int(EXPECTED_TEACHER_DIM)  # 2304
DEVICE = str(DEVICE)  # Colab GPU
USE_MIXED_PRECISION = bool(USE_MIXED_PRECISION)  # CUDA autocast (bf16 on Ampere/Hopper, fp16+GradScaler otherwise)
USE_BF16_AUTOCAST = bool(USE_BF16_AUTOCAST)  # Prefer bf16 if supported (no GradScaler needed; numerically robust on H100/A100)

# Pipeline B v3 = 17 LongCLIP-B forwards per sample (16 random tiles + 1 global @ 224 ).
# Batch sizes scale inversely vs B v2 because of the 17× per-sample tile fan-out, but the
# smaller LongCLIP-B backbone (vs ViT-L/14@336) buys back ~1.6×. Numbers tuned for A100-80 GB.
BATCH_SIZE = int(BATCH_SIZE if _ON_COLAB else 4)  # A_readout (head trainable; visual frozen)
BATCH_SIZE_VIT_SUBSEQUENT_MAX = int(BATCH_SIZE_VIT_SUBSEQUENT_MAX)        # B_last2 (last 2 ViT blocks unfrozen)
BATCH_SIZE_STAGE_C = int(BATCH_SIZE_STAGE_C)                               # C_last4 / C_last8 / C_last12 (more blocks unfrozen)
BATCH_SIZE_STAGE_D = int(BATCH_SIZE_STAGE_D)                               # D_full_decay (full ViT trainable)

# CPU-side workload is back to A's level (16 random tile crops + 1 global per sample),
# so workers help on Colab again.
# Tunable in the parameter cell (no env override required).
DATALOADER_NUM_WORKERS = int(DATALOADER_NUM_WORKERS)
DATALOADER_PREFETCH_FACTOR = int(DATALOADER_PREFETCH_FACTOR)

# Pipeline B v3.1 view layout (must match Teacher_inference_embeddings_B.ipynb exactly):
#   1 global 224x224 view + 16 random SCALE-ADAPTIVE native tiles per sample.
# Each tile's native side is computed per-image as
#   tile_px = clip(round(TILE_MAX_PX * sqrt(W*H) / TILE_SCALE_REF_PX), TILE_MIN_PX, TILE_MAX_PX)
# so larger images get bigger tiles (saturating at 224) and smaller images get smaller
# tiles (saturating at 16). The teacher computes its random pool with the exact same
# formula in nehm_pipeline.longclip_image_embeddings.LongClipBImageEmbedConfig.
N_NATIVE_RANDOM     = int(N_NATIVE_RANDOM)
TILE_PX             = int(TILE_PX)
TILE_MIN_PX         = int(TILE_MIN_PX)
TILE_MAX_PX         = int(TILE_MAX_PX)
TILE_SCALE_REF_PX   = int(TILE_SCALE_REF_PX)
CROP_SIZE           = int(CROP_SIZE)
GLOBAL_KIND         = str(GLOBAL_KIND)
PRE_CROP_MARGIN_PX  = int(PRE_CROP_MARGIN_PX)
WINDOW_SAMPLE_SEED = int(WINDOW_SAMPLE_SEED)
VAL_FRACTION = float(VAL_FRACTION)
SPLIT_SEED = int(SPLIT_SEED)

MATERIAL_LOGIT_COLUMN = str(MATERIAL_LOGIT_COLUMN)
# B v3 weak labels = HDBSCAN clusters from the new sweep on the fused 2304-D LongCLIP-B teacher.
# This trainer auto-corrects below to match the actual unique-label count in the Excel column, so
# this is just an init guess that surfaces a warning if Excel is out of sync with the latest sweep.
NUM_MATERIAL_CLASSES = int(NUM_MATERIAL_CLASSES)

EMB_LOSS_KIND = str(EMB_LOSS_KIND)
# Latent (L1) is the primary objective; CE is a probe + light regularizer. Pin CE down so
# the model cannot overfit logits at the expense of the 2304-D fused manifold.
LAMBDA_EMB_P3 = float(LAMBDA_EMB_P3)
LAMBDA_CE_P3 = float(LAMBDA_CE_P3)
LABEL_SMOOTHING_P3 = float(LABEL_SMOOTHING_P3)

RUN_PROGRESSIVE_VIT_FINETUNE = bool(RUN_PROGRESSIVE_VIT_FINETUNE)
from nehm_pipeline.checkpoint_paths import resolve_checkpoint_dir

PREFER_SAVE_CHECKPOINTS_TO_GOOGLE_DRIVE = bool(PREFER_SAVE_CHECKPOINTS_TO_GOOGLE_DRIVE)
DRIVE_PROJECT_FOLDER = str(DRIVE_PROJECT_FOLDER)
FINETUNE_CKPT_DIR = resolve_checkpoint_dir(
    PROJECT_ROOT,
    prefer_cloud=PREFER_SAVE_CHECKPOINTS_TO_GOOGLE_DRIVE,
    drive_project_folder=DRIVE_PROJECT_FOLDER,
)
FINETUNE_CKPT_BEST = FINETUNE_CKPT_DIR / str(FINETUNE_CKPT_BEST_NAME)
# Append-only JSONL: one record per epoch with full loss components (written in training cell).
TRAINING_LOG_JSONL = FINETUNE_CKPT_DIR / str(TRAINING_LOG_JSONL_NAME)

# Safety rail: refuse to silently mix a legacy 2048-D checkpoint into a 2304-D run.
ALLOW_INCOMPATIBLE_BEST_CHECKPOINT = bool(ALLOW_INCOMPATIBLE_BEST_CHECKPOINT)
if FINETUNE_CKPT_BEST.is_file():
    try:
        _best_pack = torch.load(FINETUNE_CKPT_BEST, map_location="cpu")
        _best_meta = dict(_best_pack.get("meta", {})) if isinstance(_best_pack, dict) else {}
        _best_teacher_dim = int(_best_meta.get("teacher_embed_dim") or 0)
        if _best_teacher_dim not in (0, int(EXPECTED_TEACHER_DIM)):
            _msg = (
                f"Existing best checkpoint is incompatible: {FINETUNE_CKPT_BEST} has "
                f"teacher_embed_dim={_best_teacher_dim}, expected {EXPECTED_TEACHER_DIM}. "
                "Move/rename the old .pth (or set ALLOW_INCOMPATIBLE_BEST_CHECKPOINT=True) "
                "before starting a 2304-D B v3.1 run."
            )
            if ALLOW_INCOMPATIBLE_BEST_CHECKPOINT:
                print("WARNING:", _msg)
            else:
                raise ValueError(_msg)
    except Exception as _best_ckpt_err:
        if isinstance(_best_ckpt_err, ValueError):
            raise
        print(f"Warning: could not inspect existing best checkpoint ({_best_ckpt_err}).")

# Class balancing strategy (unchanged):
USE_CLASS_BALANCED_SAMPLER = bool(USE_CLASS_BALANCED_SAMPLER)
USE_CE_CLASS_WEIGHTS = bool(USE_CE_CLASS_WEIGHTS)
SAMPLER_WEIGHT_KIND = str(SAMPLER_WEIGHT_KIND)

# Three-way student dropout split (overrides legacy single `dropout` knob):
#   input  -> applied to LongCLIP-B image emb before the decoder head
#   decoder-> inside the head (set 0.0 because decoder is the L1 target path)
#   class  -> applied to z only on the path to the classifier (CE branch only)
STUDENT_INPUT_DROPOUT = float(STUDENT_INPUT_DROPOUT)
STUDENT_DECODER_DROPOUT = float(STUDENT_DECODER_DROPOUT)
STUDENT_CLASSIFIER_DROPOUT = float(STUDENT_CLASSIFIER_DROPOUT)

# Per-row teacher noise applied during training only.
TEACHER_TARGET_NOISE_STD = float(TEACHER_TARGET_NOISE_STD)

# Stochastic Weight Averaging in D_full_decay.
USE_SWA_IN_STAGE_D = bool(USE_SWA_IN_STAGE_D)
SWA_LAST_N_EPOCHS = int(SWA_LAST_N_EPOCHS)

# A_readout acceptance gate (was: only promote A's bundle to global-best if val >= Z_embed best).
A_VIT_PROMOTE_MIN_VS_Z = bool(A_VIT_PROMOTE_MIN_VS_Z)

# Stage C (last 4/8/12 blocks): bf16 autocast is stable on H100/A100.
STAGE_C_USE_FP32 = bool(STAGE_C_USE_FP32)
# Each visual forward is at most STAGE_C_VISUAL_ENCODE_CHUNK_SIZE crops. B v3 has 17 views per
# sample, so the effective chunk shape is (BATCH_SIZE_STAGE_C × 17). Keep this small to avoid
# OOM on H100 / A100 when a stage-C unfreeze pulls extra activation memory.
STAGE_C_VISUAL_ENCODE_CHUNK_SIZE = int(STAGE_C_VISUAL_ENCODE_CHUNK_SIZE)
STAGE_D_USE_FP32 = bool(STAGE_D_USE_FP32)

# Back-compat guard: if an older tunables cell is executed, fall back to sane per-stage defaults.
EARLY_STOP_PATIENCE_VIT = dict(globals().get('EARLY_STOP_PATIENCE_VIT', {
    'B_last2': 6,
    'C_last4': 6,
    'C_last8': 8,
    'C_last12': 10,
    'D_full_decay': 12,
}))
MAX_BACKUP_KEEP = int(MAX_BACKUP_KEEP)

USE_CHANNELS_LAST = bool(USE_CHANNELS_LAST)
USE_TORCH_COMPILE_HEAD = bool(USE_TORCH_COMPILE_HEAD)
MIRROR_IMAGES_DIR = bool(MIRROR_IMAGES_DIR)

print(f"Checkpoints (.pth) directory: {FINETUNE_CKPT_DIR}")
print(f"Per-epoch log (JSONL): {TRAINING_LOG_JSONL}")
RESUME_VIT_PRIORITIES = [FINETUNE_CKPT_DIR / str(name) for name in RESUME_VIT_PRIORITY_FILENAMES]
# True: ignore every ViT finetune checkpoint above; ViT visual = LongCLIP-B weights freshly loaded.
START_FROM_BACKBONE_ONLY = bool(START_FROM_BACKBONE_ONLY)
RESUME_VIT_CANDIDATE = next((p for p in RESUME_VIT_PRIORITIES if p.is_file()), None)
RESUME_VIT_HIT = None if START_FROM_BACKBONE_ONLY else RESUME_VIT_CANDIDATE
SCHEDULE_START_STAGE = None if SCHEDULE_START_STAGE in (None, '') else str(SCHEDULE_START_STAGE)

GEOM_AUGMENT_TRAIN = bool(GEOM_AUGMENT_TRAIN)
AUGMENT_MAX_ROTATION_DEG = float(AUGMENT_MAX_ROTATION_DEG)
AUGMENT_HFLIP_P = float(AUGMENT_HFLIP_P)
AUGMENT_VFLIP_P = float(AUGMENT_VFLIP_P)

GRAD_CLIP_FINETUNE = float(GRAD_CLIP_FINETUNE)
WEIGHT_DECAY_FINETUNE = float(WEIGHT_DECAY_FINETUNE)
LAYER_DECAY_FULL = float(LAYER_DECAY_FULL)
SHOW_TQDM_BATCHES = bool(SHOW_TQDM_BATCHES)
LOG_EVERY_N_TRAIN_BATCHES = int(LOG_EVERY_N_TRAIN_BATCHES)
LOG_EVERY_N_VAL_BATCHES = int(LOG_EVERY_N_VAL_BATCHES)

EPOCHS_VIT_A = int(EPOCHS_VIT_A)
EPOCHS_VIT_A0 = int(EPOCHS_VIT_A0)
LR_READOUT_A0 = float(LR_READOUT_A0)
A_VIT_MAX_EPOCHS_PER_STAGE = int(A_VIT_MAX_EPOCHS_PER_STAGE)
A_VIT_MIN_EPOCHS_PER_STAGE = int(A_VIT_MIN_EPOCHS_PER_STAGE)
A_VIT_EARLY_STOP_PATIENCE = int(A_VIT_EARLY_STOP_PATIENCE)
EARLY_STOP_PATIENCE_EMBED = int(EARLY_STOP_PATIENCE_EMBED)
Z_EMBED_TARGET_VAL_ACC = None if Z_EMBED_TARGET_VAL_ACC in (None, '') else float(Z_EMBED_TARGET_VAL_ACC)
# Z warmup remains enabled; all subsequent stages run using the same LongCLIP-B visual tower.
RUN_EMBEDDING_STUDENT_WARMUP = bool(RUN_EMBEDDING_STUDENT_WARMUP)
EPOCHS_EMBED_WARMUP = int(EPOCHS_EMBED_WARMUP)
LR_EMBED_WARMUP = float(LR_EMBED_WARMUP)
EMBED_WARMUP_BATCH_SIZE = int(EMBED_WARMUP_BATCH_SIZE)
EMBED_WARMUP_WEIGHT_DECAY = float(EMBED_WARMUP_WEIGHT_DECAY)
EMBED_WARMUP_GRAD_CLIP = float(EMBED_WARMUP_GRAD_CLIP)
EPOCHS_VIT_B = int(EPOCHS_VIT_B)
EPOCHS_VIT_C4 = int(EPOCHS_VIT_C4)
EPOCHS_VIT_C8 = int(EPOCHS_VIT_C8)
EPOCHS_VIT_C12 = int(EPOCHS_VIT_C12)
EPOCHS_VIT_D = int(EPOCHS_VIT_D)

LR_STUDENT_A, LR_READOUT_A = float(LR_STUDENT_A), float(LR_READOUT_A)
LR_STUDENT_B, LR_READOUT_B, LR_BACKBONE_B = float(LR_STUDENT_B), float(LR_READOUT_B), float(LR_BACKBONE_B)
LR_STUDENT_C, LR_READOUT_C = float(LR_STUDENT_C), float(LR_READOUT_C)
LR_BACKBONE_C4, LR_BACKBONE_C8, LR_BACKBONE_C12 = float(LR_BACKBONE_C4), float(LR_BACKBONE_C8), float(LR_BACKBONE_C12)
LR_STUDENT_D, LR_READOUT_D, LR_BACKBONE_LAST_D = float(LR_STUDENT_D), float(LR_READOUT_D), float(LR_BACKBONE_LAST_D)

torch_device = resolve_device(DEVICE)
apply_pytorch_training_perf_for_vit_student(torch_device)
grad_scaler_amp = (
    make_cuda_grad_scaler()
    if (torch_device.type == "cuda" and USE_MIXED_PRECISION)
    else None
)
print("device:", torch_device, "| AMP:", grad_scaler_amp is not None)
_supports_bf16 = False
if torch_device.type == "cuda":
    try:
        p = torch.cuda.get_device_properties(0)
        free_b, total_b = torch.cuda.mem_get_info()
        _supports_bf16 = bool(getattr(torch.cuda, "is_bf16_supported", lambda: False)())
        print(
            f"GPU: {p.name}  VRAM free {free_b / 2**30:.1f} / {total_b / 2**30:.1f} GiB  |  "
            f"bf16_supported={_supports_bf16}  |  Pro+: prefer Run in background + A100/H100/High-RAM."
        )
    except Exception as _gpu_err:
        print(f"GPU info unavailable ({type(_gpu_err).__name__}: {_gpu_err}); restart runtime if you see device-side asserts later.")
AMP_DTYPE = torch.bfloat16 if (USE_BF16_AUTOCAST and _supports_bf16) else torch.float16


def make_vit_dataloader(dataset, *, batch_size: int, shuffle: bool) -> DataLoader:
    """Colab Pro+: workers prefetch PIL batches; persistent workers avoid fork cost each epoch."""
    kw = dict(
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=int(DATALOADER_NUM_WORKERS),
        collate_fn=collate_multiview_pil_teacher,
        pin_memory=dataloader_pin_memory(torch_device),
        drop_last=False,
    )
    if DATALOADER_NUM_WORKERS > 0:
        kw["persistent_workers"] = True
        kw["prefetch_factor"] = int(DATALOADER_PREFETCH_FACTOR)
    return DataLoader(dataset, **kw)


import pandas as pd

for _p in [IMAGE_EMB_GLOBAL, IMAGE_EMB_RANDOM, TEXT_COLUMNS_MANIFEST, MATERIAL_DB_XLSX]:
    if not _p.is_file():
        raise FileNotFoundError(
            f"Missing required B v3 input: {_p}\n"
            "  Run Notebooks/Teacher_inference_embeddings_B.ipynb first to regenerate the\n"
            f"  {EXPECTED_IMAGE_DIM}-D image (ln_post) + {EXPECTED_TEXT_EMBED_DIM}-D text (pre-proj) npys under\n"
            f"  {EMB_ROOT_B}/."
        )
print("LongCLIP-B B v3 sources (caching to local SSD on first read):")
print("  image_random :", IMAGE_EMB_RANDOM)
print("  image_global :", IMAGE_EMB_GLOBAL)
print("  text_manifest:", TEXT_COLUMNS_MANIFEST)

_xlsx_local = cache_drive_file(MATERIAL_DB_XLSX)
_img_glob_local = cache_drive_file(IMAGE_EMB_GLOBAL)
_img_rand_local = cache_drive_file(IMAGE_EMB_RANDOM)
df_sheet = pd.read_excel(_xlsx_local, sheet_name="Sheet2").reset_index(drop=True)
img_glob = np.load(_img_glob_local).astype(np.float32)
img_rand = np.load(_img_rand_local).astype(np.float32)
manifest_df = pd.read_csv(TEXT_COLUMNS_MANIFEST)

if "column_name" not in manifest_df.columns or "embedding_file" not in manifest_df.columns:
    raise ValueError(f"Unexpected text manifest format: {TEXT_COLUMNS_MANIFEST}")

# Image emb dim sanity check (catches an A or B v2 teacher pointed at by mistake).
for _name, _arr in (("global", img_glob), ("random_pooled", img_rand)):
    if int(_arr.shape[1]) != EXPECTED_IMAGE_DIM:
        raise ValueError(
            f"Image emb dim mismatch: {_name}={_arr.shape[1]}, expected {EXPECTED_IMAGE_DIM}. "
            f"Did you re-run Teacher_inference_embeddings_B.ipynb under LongCLIP-B?"
        )
if img_glob.shape[0] != img_rand.shape[0]:
    raise ValueError(
        f"Image emb row mismatch: global={img_glob.shape[0]}, random={img_rand.shape[0]}; "
        "regenerate both npys in the same teacher run."
    )

text_map = {}
text_rows_min = None
text_dim_seen = None
for _, rr in manifest_df.iterrows():
    col = str(rr["column_name"])
    _src_raw = Path(str(rr["embedding_file"]))
    if not _src_raw.is_absolute():
        _src_raw = TEXT_COLUMNS_MANIFEST.parent / _src_raw.name
    _local = cache_drive_file(_src_raw) if _src_raw.is_file() else _src_raw
    arr = np.load(str(_local)).astype(np.float32)
    if text_dim_seen is None:
        text_dim_seen = int(arr.shape[1])
    elif int(arr.shape[1]) != text_dim_seen:
        raise ValueError(f"Text emb dim mismatch for {col}: {arr.shape[1]} vs {text_dim_seen}")
    text_map[col] = arr
    text_rows_min = arr.shape[0] if text_rows_min is None else min(text_rows_min, arr.shape[0])
if text_dim_seen is not None and text_dim_seen != EXPECTED_TEXT_EMBED_DIM:
    raise ValueError(
        f"Text emb dim {text_dim_seen} != EXPECTED_TEXT_EMBED_DIM {EXPECTED_TEXT_EMBED_DIM}; "
        f"regenerate text npys under LongCLIP-B (Step B in Teacher_inference_embeddings_B.ipynb)."
    )

if MIRROR_IMAGES_DIR:
    try:
        IMAGES_DIR = cache_drive_dir(IMAGES_DIR, name="NEHM_images")
    except Exception as _mirror_err:
        print(f"Image dir mirror skipped ({_mirror_err}); falling back to Drive path.")
else:
    print(f"Images served directly from Drive (set MIRROR_IMAGES_DIR=True to pre-copy): {IMAGES_DIR}")

usable_rows = min(
    len(df_sheet), img_glob.shape[0], img_rand.shape[0],
    int(text_rows_min) if text_rows_min is not None else len(df_sheet),
)
if usable_rows <= 0:
    raise ValueError("No overlapping rows across DB/image/text embeddings")
if usable_rows < len(df_sheet):
    print(
        f"Row alignment: using first {usable_rows} rows "
        f"(db={len(df_sheet)}, img_r={img_rand.shape[0]}, img_g={img_glob.shape[0]}, text_min={text_rows_min})"
    )

df_sheet = df_sheet.iloc[:usable_rows].copy()
filenames = df_sheet["Image"].fillna("").astype(str).tolist() if "Image" in df_sheet.columns else ["" for _ in range(usable_rows)]


# Build fused teacher: image(768 ln_post) + illum(512) + avg(class/subclass)(512) + avg(desc/meta)(512) = 2304.
# B v3 image_only = L2(0.5·L2(random_pool) + 0.5·L2(global))  (Pipeline A's recipe exact).
def _l2_rows(x: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    n = np.linalg.norm(x, axis=1, keepdims=True)
    return x / np.maximum(n, eps)

_w_sum = float(IMAGE_WEIGHT_RANDOM) + float(IMAGE_WEIGHT_GLOBAL)
if _w_sum <= 0:
    raise ValueError(
        f"IMAGE_WEIGHT_RANDOM + IMAGE_WEIGHT_GLOBAL must be > 0 "
        f"(got {IMAGE_WEIGHT_RANDOM} + {IMAGE_WEIGHT_GLOBAL})."
    )
_image_blend = (
    float(IMAGE_WEIGHT_RANDOM) * _l2_rows(img_rand[:usable_rows])
    + float(IMAGE_WEIGHT_GLOBAL) * _l2_rows(img_glob[:usable_rows])
) / _w_sum
image_only = _l2_rows(_image_blend).astype(np.float32)

for bad_col in ["Image Index", "Image"]:
    text_map.pop(bad_col, None)

required_text = [
    "Illumination Modality", "Class", "Subclass", "Description", "Stain",
    "Magnification", "composition", "refractive index", "Birefringence", "Optical Sign",
]
for _c in required_text:
    if _c not in text_map:
        raise KeyError(f"Missing required text column embedding: {_c}")

for _c in list(text_map.keys()):
    text_map[_c] = text_map[_c][:usable_rows]

illum = text_map["Illumination Modality"]
group_class_subclass = ["Class", "Subclass"]
group_desc_meta = ["Description", "Stain", "Magnification", "composition", "refractive index", "Birefringence", "Optical Sign"]
composition_weight = 2.0


def _masked_group_avg(cols, weights=None):
    d = int(next(iter(text_map.values())).shape[1])
    s = np.zeros((usable_rows, d), dtype=np.float32)
    cnt = np.zeros((usable_rows, 1), dtype=np.float32)
    for col in cols:
        w = 1.0 if weights is None else float(weights.get(col, 1.0))
        valid = (
            df_sheet[col].fillna("").astype(str)
            .map(lambda x: x.strip() != "" and x.strip().lower() != "nan")
            .to_numpy(dtype=np.float32).reshape(usable_rows, 1)
        ) if col in df_sheet.columns else np.zeros((usable_rows, 1), dtype=np.float32)
        vw = valid * w
        s += text_map[col] * vw
        cnt += vw
    out = (s / np.maximum(cnt, 1.0)).astype(np.float32)
    out[cnt[:, 0] == 0.0] = 0.0
    return out

avg_class_subclass = _masked_group_avg(group_class_subclass)
avg_desc_meta = _masked_group_avg(group_desc_meta, weights={"composition": composition_weight})

fused = np.concatenate([image_only, illum, avg_class_subclass, avg_desc_meta], axis=1).astype(np.float32)
if int(fused.shape[1]) != EXPECTED_TEACHER_DIM:
    raise ValueError(
        f"Expected fused dim {EXPECTED_TEACHER_DIM}, got {fused.shape[1]} "
        f"(image_only={image_only.shape[1]} + 3 × text {next(iter(text_map.values())).shape[1]})"
    )

_IMG_D = int(image_only.shape[1])  # teacher image block width (must match EXPECTED_IMAGE_DIM in this contract)
_teach_dim = int(fused.shape[1])
z_embed_image_input = fused[:, :_IMG_D].astype(np.float32, copy=False)
print(f"Teacher fused shape: {fused.shape} | image block {_IMG_D} | teacher_dim {_teach_dim}")

N = len(filenames)
material_targets = load_material_logits_from_excel(
    _xlsx_local,
    n_rows=N,
    column=MATERIAL_LOGIT_COLUMN,
    sheet_name="Sheet2",
)
lab_ix = np.where(material_targets >= 0)[0]
y_lab = material_targets[lab_ix]
if len(y_lab) > 0:
    k_data = int(y_lab.max()) + 1
    n_unique = int(np.unique(y_lab).size)
    if k_data != int(NUM_MATERIAL_CLASSES):
        print(
            f"Auto-correcting NUM_MATERIAL_CLASSES: {NUM_MATERIAL_CLASSES} -> {k_data} "
            f"(unique labels seen: {n_unique})."
        )
        NUM_MATERIAL_CLASSES = k_data
    if n_unique != k_data:
        print(
            f"  Note: max+1 ({k_data}) != unique ({n_unique}); label space is sparse. "
            f"Classifier still sized to {k_data} so all integer ids fit; remap to contiguous if you need balanced softmax."
        )
train_ix, val_ix = train_val_split_rows(lab_ix, y_lab, VAL_FRACTION, SPLIT_SEED)
print(f"labeled train/val: {len(train_ix)} {len(val_ix)} | {MATERIAL_LOGIT_COLUMN=!r} K={NUM_MATERIAL_CLASSES}")


# ----- LongCLIP-B (ViT-B/16, 512-D image + 248-token text). The package is vendored as a
# relative-import package under PROJECT_ROOT/LongCLIP/, so we load it via sys.path + importlib
# the same way the teacher / inference notebooks do.
if not LONGCLIP_ROOT.is_dir():
    _cands = '\n  '.join(str(p) for p in _longclip_root_candidates)
    raise FileNotFoundError(
        f"LongCLIP root not found. Tried:\n  {_cands}\n"
        f"Set LONGCLIP_ROOT_REL in the tunables cell to your actual LongCLIP folder."
    )


def _load_longclip_module():
    """Robust LongCLIP package loader (mirrors the teacher / inference notebooks)."""
    import importlib
    import importlib.util

    pkg_dir = LONGCLIP_ROOT / "model"
    init_py = pkg_dir / "__init__.py"
    longclip_py = pkg_dir / "longclip.py"
    model_longclip_py = pkg_dir / "model_longclip.py"
    simple_tok_py = pkg_dir / "simple_tokenizer.py"

    missing = [p for p in (pkg_dir, init_py, longclip_py, model_longclip_py, simple_tok_py) if not p.exists()]
    if missing:
        listing = "\n    ".join(str(p) for p in pkg_dir.iterdir()) if pkg_dir.is_dir() else "(missing)"
        raise FileNotFoundError(
            "LongCLIP package incomplete on Drive. Missing:\n  "
            + "\n  ".join(str(p) for p in missing)
            + f"\nContents of {pkg_dir}:\n    {listing}\n"
            + "Re-upload the entire LongCLIP/model/ directory (especially __init__.py and model_longclip.py)."
        )

    if str(LONGCLIP_ROOT) not in sys.path:
        sys.path.insert(0, str(LONGCLIP_ROOT))
    for k in [k for k in list(sys.modules) if k == "model" or k.startswith("model.")]:
        del sys.modules[k]

    try:
        from model import longclip as _lc  # type: ignore
        return _lc
    except Exception as _simple_err:
        print(f"  (simple `from model import longclip` failed: {_simple_err}; using importlib package loader)")

    pkg_spec = importlib.util.spec_from_file_location(
        "model", str(init_py), submodule_search_locations=[str(pkg_dir)],
    )
    if pkg_spec is None or pkg_spec.loader is None:
        raise ImportError(f"Failed to create package spec for {init_py}")
    pkg_mod = importlib.util.module_from_spec(pkg_spec)
    sys.modules["model"] = pkg_mod
    pkg_spec.loader.exec_module(pkg_mod)

    sub_spec = importlib.util.spec_from_file_location("model.longclip", str(longclip_py))
    if sub_spec is None or sub_spec.loader is None:
        raise ImportError(f"Failed to create submodule spec for {longclip_py}")
    sub_mod = importlib.util.module_from_spec(sub_spec)
    sys.modules["model.longclip"] = sub_mod
    sub_spec.loader.exec_module(sub_mod)
    return sub_mod


longclip_module = _load_longclip_module()
print(f"LongCLIP loaded from: {LONGCLIP_ROOT / 'model' / 'longclip.py'}")

if not LONGCLIP_CHECKPOINT.is_file():
    _ckpt_cands = '\n  '.join(str(p) for p in _longclip_ckpt_candidates)
    raise FileNotFoundError(
        f"LongCLIP checkpoint not found. Tried:\n  {_ckpt_cands}\n"
        f"Set LONGCLIP_CHECKPOINT_REL in the tunables cell to the correct checkpoint path."
    )

_longclip_local = cache_drive_file(LONGCLIP_CHECKPOINT)
clip_model, _preprocess_with_normalize = longclip_module.load(str(_longclip_local), device=str(torch_device))
clip_model = clip_model.float().to(torch_device)

# Strip torchvision's Normalize(mean, std) so RGB stays linear in [0, 1] (NEHM framework default;
# preserves microscopy-relevant per-channel intensity ratios). Teacher / inference notebooks do
# the same; this is what keeps every stage of the pipeline aligned on the same pixel scale.
preprocess = clip_visual_preprocess_without_channel_normalize(_preprocess_with_normalize)
print("LongCLIP-B preprocess: Normalize stripped (linear RGB in [0, 1])")

visual = clip_model.visual
visual.train()


# Forward-probe the visual tower: teacher npys may be **512-D** (full ``visual``, after ``proj``)
# or **768-D** (ViT-B/16 CLS after ``ln_post``, before ``proj``). Match ``_IMG_D`` from disk.
def _probe_visual_student_image_dim(
    visual_module: torch.nn.Module,
    *,
    device: torch.device,
    image_emb_dim: int,
) -> int:
    from nehm_pipeline.vit_ln_post_utils import vit_visual_ln_post_forward

    res = int(getattr(visual_module, "input_resolution", CROP_SIZE))
    sample = torch.zeros(1, 3, res, res, device=device, dtype=next(visual_module.parameters()).dtype)
    was_training = visual_module.training
    visual_module.eval()
    try:
        with torch.inference_mode():
            full_out = visual_module(sample)
            ln_post_out = vit_visual_ln_post_forward(visual_module, sample)
    finally:
        visual_module.train(was_training)
    d_full = int(full_out.shape[-1])
    d_ln = int(ln_post_out.shape[-1])
    want = int(image_emb_dim)
    if want == d_full:
        return d_full
    if want == d_ln:
        return d_ln
    raise ValueError(
        f"Teacher image emb dim is {want} but this LongCLIP visual gives {d_full}-D (after proj) "
        f"or {d_ln}-D (ln_post CLS). Re-run Teacher_inference_embeddings_B.ipynb or fix "
        f"TEACHER_NPY_PATHS / LONGCLIP_CHECKPOINT so image npys and weights agree."
    )


visual_dim = _probe_visual_student_image_dim(visual, device=torch_device, image_emb_dim=int(_IMG_D))
print(
    f"LongCLIP-B visual probe ({CLIP_MODEL}): output_dim={visual_dim} (matches teacher image block {_IMG_D})"
)

# ViT stages must pool the same width as the teacher image block: 512-D after ``visual.proj`` or
# 768-D ln_post CLS. Z_embed already uses ``fused[:, :_IMG_D]``.
VIT_STAGE_USE_LN_POST_ENCODING = int(_IMG_D) == 768
print(
    f"ViT fine-tune encoding: {'ln_post CLS (768)' if VIT_STAGE_USE_LN_POST_ENCODING else 'full visual after proj (512)'} "
    f"(teacher image block {_IMG_D}-D)"
)

if USE_CHANNELS_LAST and torch_device.type == "cuda":
    visual = visual.to(memory_format=torch.channels_last)
    print("Visual: memory_format=channels_last")

import inspect as _inspect_dropout
import types as _types_dropout

edim = int(fused.shape[1])

# Detect whether the loaded pipeline copy supports the three-way dropout knobs (newer file).
_student_init_params = _inspect_dropout.signature(ImageEmbeddingStudent.__init__).parameters
_student_supports_split_dropout = "input_dropout" in _student_init_params

# B v3.1 student head: _IMG_D -> 1024 -> 1536 -> edim (decoder), then Linear(edim -> K) for CE.
# _IMG_D and edim are inferred from the loaded teacher artifacts and validated above.
# decoder_hidden keeps a smooth expansion toward the fused teacher target without overshooting.
DECODER_HIDDEN_B_V3 = (1024, 1536)

if _student_supports_split_dropout:
    student = ImageEmbeddingStudent(
        image_dim=_IMG_D,
        num_classes=NUM_MATERIAL_CLASSES,
        embed_dim=edim,
        decoder_hidden=DECODER_HIDDEN_B_V3,
        head_style="decoder",
        input_dropout=STUDENT_INPUT_DROPOUT,
        decoder_dropout=STUDENT_DECODER_DROPOUT,
        classifier_dropout=STUDENT_CLASSIFIER_DROPOUT,
    ).to(torch_device)
    _dropout_source = "native (pipeline)"
else:
    student = ImageEmbeddingStudent(
        image_dim=_IMG_D,
        num_classes=NUM_MATERIAL_CLASSES,
        embed_dim=edim,
        decoder_hidden=DECODER_HIDDEN_B_V3,
        head_style="decoder",
        dropout=float(STUDENT_DECODER_DROPOUT),
    ).to(torch_device)
    student.input_drop = (
        torch.nn.Dropout(float(STUDENT_INPUT_DROPOUT)) if float(STUDENT_INPUT_DROPOUT) > 0 else torch.nn.Identity()
    ).to(torch_device)
    student.classifier_drop = (
        torch.nn.Dropout(float(STUDENT_CLASSIFIER_DROPOUT)) if float(STUDENT_CLASSIFIER_DROPOUT) > 0 else torch.nn.Identity()
    ).to(torch_device)

    def _patched_student_forward(self, image_emb):
        f = image_emb.float()
        f = self.input_drop(f)
        z = self.head(f)
        logits = self.classifier(self.classifier_drop(z))
        return z, logits

    student.forward = _types_dropout.MethodType(_patched_student_forward, student)
    _dropout_source = "polyfill (Drive pipeline is older; rebuild forward)"

print(
    f"Student head: image_dim={_IMG_D} -> embed_dim={edim} | classes={NUM_MATERIAL_CLASSES} "
    f"| dropout(input/decoder/classifier)={STUDENT_INPUT_DROPOUT}/{STUDENT_DECODER_DROPOUT}/{STUDENT_CLASSIFIER_DROPOUT} "
    f"[{_dropout_source}]"
)

if USE_TORCH_COMPILE_HEAD and hasattr(torch, "compile"):
    try:
        student.head = torch.compile(student.head, mode="reduce-overhead", dynamic=False)
        print("Student head: torch.compile(mode='reduce-overhead') enabled")
    except Exception as _comp_err:
        print(f"torch.compile(student.head) skipped: {_comp_err}")


def _student_head_shape_map(model: torch.nn.Module) -> str:
    dims = [int(_IMG_D)]
    if hasattr(model.head, "blocks"):
        for blk in model.head.blocks:
            if hasattr(blk, "linear"):
                dims.append(int(blk.linear.out_features))
            elif isinstance(blk, torch.nn.Linear):
                dims.append(int(blk.out_features))
    elif isinstance(model.head, torch.nn.Sequential):
        for lyr in model.head:
            if isinstance(lyr, torch.nn.Linear):
                dims.append(int(lyr.out_features))
    c_in = int(model.classifier.in_features)
    c_out = int(model.classifier.out_features)
    if not dims or dims[-1] != c_in:
        dims.append(c_in)
    chain = " -> ".join(str(d) for d in dims)
    return f"{chain} -> {c_out} logits"


print("Student dimensional flow (expansion/contraction):")
print("  ", _student_head_shape_map(student))

full_model = ClipVisualStudent(visual, student)

print("--- ViT fine-tune resume (first matching file on disk) ---")
if START_FROM_BACKBONE_ONLY:
    print(
        f"  START_FROM_BACKBONE_ONLY=True → ViT visual = freshly loaded {CLIP_MODEL} "
        "(finetuned .pth files ignored)."
    )
    if RESUME_VIT_CANDIDATE is not None:
        print(f"  (ignored: {RESUME_VIT_CANDIDATE})")
for _cand in RESUME_VIT_PRIORITIES:
    _ok = _cand.is_file()
    _mark = "  <-- using this" if (_ok and RESUME_VIT_HIT is not None and _cand == RESUME_VIT_HIT) else ""
    print(f"  {'found' if _ok else 'absent':6}  {_cand.name}{_mark}")

if RESUME_VIT_HIT is not None:
    try:
        load_clip_visual_student_checkpoint_skip_classifier(
            RESUME_VIT_HIT, full_model, map_location=torch_device
        )
        print(
            f"Loaded ViT visual + student decoder from: {RESUME_VIT_HIT.name}\n"
            f"  (Classifier random-init for K={NUM_MATERIAL_CLASSES} {MATERIAL_LOGIT_COLUMN!r}.)"
        )
    except (RuntimeError, ValueError, KeyError) as _resume_err:
        print(
            f"Resume failed ({type(_resume_err).__name__}: {_resume_err}); falling back to fresh {CLIP_MODEL} weights."
        )
else:
    print(f"No ViT .pth selected; ViT visual = freshly loaded {CLIP_MODEL}.")
    if STUDENT_CKPT.is_file():
        print("Loading student decoder from STUDENT_CKPT.")
        try:
            pack = torch.load(STUDENT_CKPT, map_location=torch_device)
            sd = dict(pack["student_state_dict"])
            for k in list(sd.keys()):
                if k.startswith("classifier."):
                    del sd[k]
            miss, unexp = student.load_state_dict(sd, strict=False)
            bad = [k for k in miss if not k.startswith("classifier.")]
            if bad or unexp:
                print(f"  warning: partial load (missing={bad!r}, unexpected={unexp!r}); continuing.")
            print(f"  Decoder weights: {STUDENT_CKPT.name}")
        except Exception as _stud_err:
            print(f"  STUDENT_CKPT load failed ({_stud_err}); using random init.")
    else:
        print("STUDENT_CKPT missing; starting student from random init.")

meta = {
    "n_material_classes": NUM_MATERIAL_CLASSES,
    "material_logit_column": MATERIAL_LOGIT_COLUMN,
    "teacher_embed_dim": edim,
}

from torch.utils.data import WeightedRandomSampler


class _TeacherNoiseDatasetWrapper(torch.utils.data.Dataset):
    """Adds per-row Gaussian noise (scaled by per-dim teacher std) to the teacher target."""

    def __init__(self, base, *, noise_std: float, t_scale: np.ndarray | None) -> None:
        self.base = base
        self.noise_std = float(noise_std)
        self.t_scale = None if t_scale is None else np.asarray(t_scale, dtype=np.float32)

    def __len__(self):
        return len(self.base)

    def __getitem__(self, idx):
        item = self.base[idx]
        if self.noise_std <= 0 or self.t_scale is None:
            return item
        if not (isinstance(item, tuple) and len(item) == 3):
            return item
        a, t, y = item
        noise = np.random.randn(*self.t_scale.shape).astype(np.float32) * self.t_scale * self.noise_std
        if isinstance(t, torch.Tensor):
            t = t + torch.from_numpy(noise).to(t.dtype)
        else:
            t = (np.asarray(t, dtype=np.float32) + noise).astype(np.float32, copy=False)
        return a, t, y


def make_vit_dataloader_v2(dataset, *, batch_size: int, shuffle: bool, sampler=None) -> DataLoader:
    """Sampler-aware variant: pass a WeightedRandomSampler for class-balanced training."""
    kw = dict(
        batch_size=batch_size,
        num_workers=int(DATALOADER_NUM_WORKERS),
        collate_fn=collate_multiview_pil_teacher,
        pin_memory=dataloader_pin_memory(torch_device),
        drop_last=False,
    )
    if sampler is not None:
        kw["sampler"] = sampler
    else:
        kw["shuffle"] = shuffle
    if DATALOADER_NUM_WORKERS > 0:
        kw["persistent_workers"] = True
        kw["prefetch_factor"] = int(DATALOADER_PREFETCH_FACTOR)
    return DataLoader(dataset, **kw)


def build_class_balanced_sampler(y_train: np.ndarray, *, kind: str = "inv_sqrt") -> tuple[WeightedRandomSampler | None, dict]:
    y_train = np.asarray(y_train, dtype=np.int64).ravel()
    if kind == "none" or len(y_train) == 0:
        return None, {"kind": kind}
    K = int(y_train.max()) + 1 if y_train.size else 0
    counts = np.bincount(y_train, minlength=max(K, 1)).astype(np.float64)
    counts = np.where(counts > 0, counts, 1.0)
    if kind == "inv":
        per_class_w = 1.0 / counts
    elif kind == "inv_sqrt":
        per_class_w = 1.0 / np.sqrt(counts)
    else:
        raise ValueError(f"SAMPLER_WEIGHT_KIND must be 'inv' / 'inv_sqrt' / 'none', got {kind!r}")
    w = per_class_w[y_train]
    w = w / max(float(w.sum()), 1e-12)
    sampler = WeightedRandomSampler(
        weights=torch.as_tensor(w, dtype=torch.double),
        num_samples=len(y_train),
        replacement=True,
    )
    info = {
        "kind": kind,
        "n_classes_observed": int((counts > 0).sum()),
        "min_count": int(counts.min()),
        "max_count": int(counts.max()),
        "weight_min": float(w.min()),
        "weight_max": float(w.max()),
    }
    return sampler, info


import inspect as _inspect_b
_lmd_params_b = _inspect_b.signature(LabeledMultiviewFinetuneDataset.__init__).parameters
for _need in ("tile_px", "global_kind", "pre_crop_margin_px"):
    if _need not in _lmd_params_b:
        raise RuntimeError(
            f"LabeledMultiviewFinetuneDataset on Drive lacks `{_need}`. "
            "Upload the latest nehm_pipeline/clip_vit_multiview_finetune.py from NEHM_Stable to Drive — "
            "Pipeline B v3.1 requires it (see README_B.md)."
        )
for _need in ("tile_min_px", "tile_max_px", "tile_scale_ref_px"):
    if _need not in _lmd_params_b:
        raise RuntimeError(
            f"LabeledMultiviewFinetuneDataset on Drive lacks `{_need}`. "
            "Upload the latest nehm_pipeline/clip_vit_multiview_finetune.py from NEHM_Stable to Drive — "
            "Pipeline B v3.1 (scale-adaptive multiview) requires it (see README_B.md)."
        )
if N_NATIVE_RANDOM == 0:
    print(
        f"B view layout (single-view ablation mode): "
        f"1 global view only ({GLOBAL_KIND} resize, {CROP_SIZE}x{CROP_SIZE}, "
        f"pre_crop_margin_px={PRE_CROP_MARGIN_PX}); no random tiles."
    )
elif int(TILE_PX) > 0:
    print(
        f"B v3 view layout (must match Teacher_inference_embeddings_B.ipynb): "
        f"1 global ({GLOBAL_KIND} resize, {CROP_SIZE}x{CROP_SIZE}, "
        f"pre_crop_margin_px={PRE_CROP_MARGIN_PX}) + "
        f"{N_NATIVE_RANDOM} random {TILE_PX}x{TILE_PX} fixed-size native tiles "
        f"= {1 + N_NATIVE_RANDOM} views/sample."
    )
else:
    print(
        f"B v3.1 view layout (must match Teacher_inference_embeddings_B.ipynb): "
        f"1 global ({GLOBAL_KIND} resize, {CROP_SIZE}x{CROP_SIZE}, "
        f"pre_crop_margin_px={PRE_CROP_MARGIN_PX}) + "
        f"{N_NATIVE_RANDOM} random scale-adaptive native tiles "
        f"[{TILE_MIN_PX}-{TILE_MAX_PX}px @ ref={TILE_SCALE_REF_PX}] "
        f"= {1 + N_NATIVE_RANDOM} views/sample."
    )

_teacher_dim_train = int(fused.shape[1])
_teacher_train_subset = fused[train_ix]
_teacher_scale_full = _teacher_train_subset.std(axis=0).astype(np.float32) if TEACHER_TARGET_NOISE_STD > 0 else None
_teacher_scale_img_only = (
    z_embed_image_input[train_ix].std(axis=0).astype(np.float32)
    if TEACHER_TARGET_NOISE_STD > 0 and isinstance(z_embed_image_input, np.ndarray)
    else None
)

ds_tr_base = LabeledMultiviewFinetuneDataset(
    train_ix,
    material_targets,
    filenames,
    IMAGES_DIR,
    n_native_random=N_NATIVE_RANDOM,
    crop_size=CROP_SIZE,
    fused_teacher=fused,
    window_sample_seed=WINDOW_SAMPLE_SEED,
    geom_augment=GEOM_AUGMENT_TRAIN,
    augment_max_rotation_deg=AUGMENT_MAX_ROTATION_DEG,
    augment_hflip_p=AUGMENT_HFLIP_P,
    augment_vflip_p=AUGMENT_VFLIP_P,
    tile_px=TILE_PX,
    global_kind=GLOBAL_KIND,
    pre_crop_margin_px=PRE_CROP_MARGIN_PX,
    tile_min_px=TILE_MIN_PX,
    tile_max_px=TILE_MAX_PX,
    tile_scale_ref_px=TILE_SCALE_REF_PX,
)
ds_tr = _TeacherNoiseDatasetWrapper(
    ds_tr_base,
    noise_std=TEACHER_TARGET_NOISE_STD,
    t_scale=_teacher_train_subset.std(axis=0).astype(np.float32) if TEACHER_TARGET_NOISE_STD > 0 else None,
)
ds_va = LabeledMultiviewFinetuneDataset(
    val_ix,
    material_targets,
    filenames,
    IMAGES_DIR,
    n_native_random=N_NATIVE_RANDOM,
    crop_size=CROP_SIZE,
    fused_teacher=fused,
    window_sample_seed=WINDOW_SAMPLE_SEED,
    geom_augment=False,
    tile_px=TILE_PX,
    global_kind=GLOBAL_KIND,
    pre_crop_margin_px=PRE_CROP_MARGIN_PX,
    tile_min_px=TILE_MIN_PX,
    tile_max_px=TILE_MAX_PX,
    tile_scale_ref_px=TILE_SCALE_REF_PX,
)

y_train_arr = np.asarray(material_targets[train_ix], dtype=np.int64)
if USE_CLASS_BALANCED_SAMPLER and SAMPLER_WEIGHT_KIND != "none":
    train_sampler, _sampler_info = build_class_balanced_sampler(y_train_arr, kind=SAMPLER_WEIGHT_KIND)
    print(
        f"WeightedRandomSampler ({_sampler_info['kind']}): "
        f"classes={_sampler_info.get('n_classes_observed')} "
        f"counts[min..max]=[{_sampler_info.get('min_count')}..{_sampler_info.get('max_count')}] "
        f"weights[min..max]=[{_sampler_info.get('weight_min'):.2e}..{_sampler_info.get('weight_max'):.2e}]"
    )
else:
    train_sampler = None
    print("Sampler: shuffle only (USE_CLASS_BALANCED_SAMPLER=False or SAMPLER_WEIGHT_KIND=='none')")

ld_tr = make_vit_dataloader_v2(ds_tr, batch_size=BATCH_SIZE, shuffle=True, sampler=train_sampler)
ld_va = make_vit_dataloader_v2(ds_va, batch_size=BATCH_SIZE, shuffle=False, sampler=None)
print(
    f"DataLoader: num_workers={DATALOADER_NUM_WORKERS}  "
    f"batches train/val per epoch={len(ld_tr)}/{len(ld_va)}  batch_size={BATCH_SIZE}  "
    f"sampler={'on' if train_sampler is not None else 'off'}  teacher_noise_std={TEACHER_TARGET_NOISE_STD}"
)

if USE_CE_CLASS_WEIGHTS:
    raise ValueError(
        "USE_CE_CLASS_WEIGHTS=True is not the right knob here; prefer USE_CLASS_BALANCED_SAMPLER "
        "so the L1 latent loss also sees a balanced teacher distribution."
    )
print(
    f"class balance: sampler={'inv_sqrt' if (USE_CLASS_BALANCED_SAMPLER and SAMPLER_WEIGHT_KIND!='none') else 'off'}  |  "
    f"train: unweighted CE  |  val: unbalanced (honest generalization metric)"
)


def write_run_config_manifest() -> Path:
    """Persist the full run config next to the checkpoints so .pth files stay traceable."""
    cfg = {
        "ts_utc": _dt.now(_tz.utc).isoformat(),
        "global_seed": int(GLOBAL_SEED),
        "pipeline_variant": "B_v3.1" if int(TILE_PX) <= 0 else "B_v3",
        "clip_model": CLIP_MODEL,
        "longclip_root": str(LONGCLIP_ROOT),
        "longclip_checkpoint": str(LONGCLIP_CHECKPOINT),
        "expected_image_dim": int(EXPECTED_IMAGE_DIM),
        "expected_teacher_dim": int(EXPECTED_TEACHER_DIM),
        "image_emb_random": str(IMAGE_EMB_RANDOM),
        "image_emb_global": str(IMAGE_EMB_GLOBAL),
        "image_weight_random": float(IMAGE_WEIGHT_RANDOM),
        "image_weight_global": float(IMAGE_WEIGHT_GLOBAL),
        "text_columns_manifest": str(TEXT_COLUMNS_MANIFEST),
        "material_db_xlsx": str(MATERIAL_DB_XLSX),
        "material_logit_column": MATERIAL_LOGIT_COLUMN,
        "num_material_classes": int(NUM_MATERIAL_CLASSES),
        "image_dim": int(_IMG_D),
        "teacher_dim": int(edim),
        "decoder_hidden": list(DECODER_HIDDEN_B_V3),
        "n_native_random": int(N_NATIVE_RANDOM),
        "tile_px": None if TILE_PX is None else int(TILE_PX),
        "tile_min_px": int(TILE_MIN_PX),
        "tile_max_px": int(TILE_MAX_PX),
        "tile_scale_ref_px": int(TILE_SCALE_REF_PX),
        "crop_size": int(CROP_SIZE),
        "global_kind": str(GLOBAL_KIND),
        "pre_crop_margin_px": int(PRE_CROP_MARGIN_PX),
        "val_fraction": float(VAL_FRACTION),
        "split_seed": int(SPLIT_SEED),
        "window_sample_seed": int(WINDOW_SAMPLE_SEED),
        "batch_size_a": int(BATCH_SIZE),
        "batch_size_subseq_max": int(BATCH_SIZE_VIT_SUBSEQUENT_MAX),
        "batch_size_stage_c": int(BATCH_SIZE_STAGE_C),
        "batch_size_stage_d": int(BATCH_SIZE_STAGE_D),
        "stage_c_use_fp32": bool(STAGE_C_USE_FP32),
        "stage_d_use_fp32": bool(STAGE_D_USE_FP32),
        "stage_c_visual_encode_chunk_size": int(STAGE_C_VISUAL_ENCODE_CHUNK_SIZE),
        "use_mixed_precision": bool(USE_MIXED_PRECISION),
        "use_bf16_autocast": bool(USE_BF16_AUTOCAST),
        "amp_dtype": str(AMP_DTYPE).replace("torch.", ""),
        "use_channels_last": bool(USE_CHANNELS_LAST),
        "use_torch_compile_head": bool(USE_TORCH_COMPILE_HEAD),
        "dataloader_num_workers": int(DATALOADER_NUM_WORKERS),
        "dataloader_prefetch_factor": int(DATALOADER_PREFETCH_FACTOR),
        "geom_augment_train": bool(GEOM_AUGMENT_TRAIN),
        "augment_max_rotation_deg": float(AUGMENT_MAX_ROTATION_DEG),
        "augment_hflip_p": float(AUGMENT_HFLIP_P),
        "augment_vflip_p": float(AUGMENT_VFLIP_P),
        "lambda_emb": float(LAMBDA_EMB_P3),
        "lambda_ce": float(LAMBDA_CE_P3),
        "label_smoothing": float(LABEL_SMOOTHING_P3),
        "emb_loss_kind": EMB_LOSS_KIND,
        "weight_decay_finetune": float(WEIGHT_DECAY_FINETUNE),
        "grad_clip_finetune": float(GRAD_CLIP_FINETUNE),
        "layer_decay_full": float(LAYER_DECAY_FULL),
        "early_stop_patience_vit": EARLY_STOP_PATIENCE_VIT,
        "early_stop_patience_embed": EARLY_STOP_PATIENCE_EMBED,
        "a_vit_max_epochs": int(A_VIT_MAX_EPOCHS_PER_STAGE),
        "a_vit_min_epochs": int(A_VIT_MIN_EPOCHS_PER_STAGE),
        "a_vit_early_stop_patience": A_VIT_EARLY_STOP_PATIENCE,
        "epochs_embed_warmup": int(EPOCHS_EMBED_WARMUP),
        "epochs_vit": {
            "B_last2": int(EPOCHS_VIT_B),
            "C_last4": int(EPOCHS_VIT_C4),
            "C_last8": int(EPOCHS_VIT_C8),
            "C_last12": int(EPOCHS_VIT_C12),
            "D_full_decay": int(EPOCHS_VIT_D),
        },
        "lr": {
            "embed_warmup": float(LR_EMBED_WARMUP),
            "A": {"student": LR_STUDENT_A, "readout": LR_READOUT_A},
            "B": {"student": LR_STUDENT_B, "readout": LR_READOUT_B, "backbone": LR_BACKBONE_B},
            "C": {
                "student": LR_STUDENT_C,
                "readout": LR_READOUT_C,
                "backbone_4": LR_BACKBONE_C4,
                "backbone_8": LR_BACKBONE_C8,
                "backbone_12": LR_BACKBONE_C12,
            },
            "D": {"student": LR_STUDENT_D, "readout": LR_READOUT_D, "backbone_last": LR_BACKBONE_LAST_D},
        },
        "schedule_start_stage": SCHEDULE_START_STAGE,
        "start_from_backbone_only": bool(START_FROM_BACKBONE_ONLY),
        "resume_vit_hit": str(RESUME_VIT_HIT) if RESUME_VIT_HIT else None,
        "max_backup_keep": int(MAX_BACKUP_KEEP),
        "row_count": int(usable_rows),
        "train_size": int(len(train_ix)),
        "val_size": int(len(val_ix)),
        "use_class_balanced_sampler": bool(USE_CLASS_BALANCED_SAMPLER),
        "sampler_weight_kind": SAMPLER_WEIGHT_KIND,
        "student_input_dropout": float(STUDENT_INPUT_DROPOUT),
        "student_decoder_dropout": float(STUDENT_DECODER_DROPOUT),
        "student_classifier_dropout": float(STUDENT_CLASSIFIER_DROPOUT),
        "teacher_target_noise_std": float(TEACHER_TARGET_NOISE_STD),
        "use_swa_in_stage_d": bool(USE_SWA_IN_STAGE_D),
        "swa_last_n_epochs": int(SWA_LAST_N_EPOCHS),
        "a_vit_promote_min_vs_z": bool(A_VIT_PROMOTE_MIN_VS_Z),
    }
    out = FINETUNE_CKPT_DIR / "vit_finetune_run_config_B.json"
    out.parent.mkdir(parents=True, exist_ok=True)
    with open(out, "w", encoding="utf-8") as f:
        _json.dump(cfg, f, indent=2)
    return out


_run_config_path = write_run_config_manifest()
print(f"Run config manifest: {_run_config_path}")


In [ ]:
# Progressive training: **Z_embed** pretrains the student head on LongCLIP-B image embeddings,
# with fused image+text (2304-D under B v3.1) as teacher target. Optionally followed by ViT stages if enabled.
import copy
import importlib
import inspect
import json
import math
import shutil
from datetime import datetime, timezone
from pathlib import Path as CheckpointPath

importlib.reload(vfin)
importlib.reload(sc)
from nehm_pipeline.clip_vit_multiview_finetune import (
    build_adamw_vit_finetune,
    freeze_clip_visual,
    vit_finetune_run_epoch,
)
from nehm_pipeline.clip_visual_student import unified_finetune_checkpoint_dict

_vit_amp_warned = False


def _append_training_epoch_log(row: dict) -> None:
    """Append one JSON object per line to TRAINING_LOG_JSONL (full epoch metrics)."""
    row = {**row, "ts_utc": datetime.now(timezone.utc).isoformat()}
    TRAINING_LOG_JSONL.parent.mkdir(parents=True, exist_ok=True)
    with open(TRAINING_LOG_JSONL, "a", encoding="utf-8") as f:
        f.write(json.dumps(row) + "\n")


def _vit_finetune_amp_kwargs(*, for_training: bool, stage_name: str) -> dict:
    """AMP vs FP32 by stage.
    Uses bf16 (no GradScaler) only if the Drive helper supports ``amp_dtype``;
    otherwise falls back to fp16 + GradScaler (which the helper requires for training).
    """
    global _vit_amp_warned
    use_fp32_stage = (stage_name.startswith("C_") and STAGE_C_USE_FP32) or (
        stage_name == "D_full_decay" and STAGE_D_USE_FP32
    )
    if use_fp32_stage:
        return {}
    if not USE_MIXED_PRECISION:
        return {}
    sig = inspect.signature(vit_finetune_run_epoch).parameters
    if "use_amp" not in sig:
        if not _vit_amp_warned:
            print(
                "Warning: Drive nehm_pipeline/clip_vit_multiview_finetune.py has no AMP — "
                "copy the latest file from your repo, or set USE_MIXED_PRECISION=False.",
                flush=True,
            )
            _vit_amp_warned = True
        return {}
    kw: dict = {"use_amp": True}
    # bf16 path requires the helper to accept amp_dtype AND tolerate grad_scaler=None for training.
    bf16_ok = (AMP_DTYPE is torch.bfloat16) and ("amp_dtype" in sig)
    if bf16_ok:
        kw["amp_dtype"] = torch.bfloat16
        kw["grad_scaler"] = None
    else:
        # fp16 + GradScaler. Mandatory for training; eval can pass None.
        if AMP_DTYPE is torch.bfloat16 and not _vit_amp_warned:
            print(
                "Note: Drive clip_vit_multiview_finetune.py has no `amp_dtype` arg; "
                "falling back to fp16 + GradScaler. Update the file on Drive to enable bf16.",
                flush=True,
            )
            _vit_amp_warned = True
        if "amp_dtype" in sig:
            kw["amp_dtype"] = torch.float16
        kw["grad_scaler"] = grad_scaler_amp if for_training else None
    return kw


def backup_checkpoint_if_exists(path) -> None:
    """If ``path`` already exists, copy it to ``stem.bak_YYYYMMDD_HHMMSS.pth`` (never clobber).
    Old backups beyond MAX_BACKUP_KEEP get pruned so Drive doesn't fill up."""
    path = CheckpointPath(path)
    if path.is_file():
        ts = datetime.now().strftime("%Y%m%d_%H%M%S")
        bak = path.parent / f"{path.stem}.bak_{ts}{path.suffix}"
        shutil.copy2(path, bak)
        print(f"  (preserved previous → {bak.name})", flush=True)
    removed = prune_old_backups(path.parent, stem_prefix=path.stem, keep=MAX_BACKUP_KEEP)
    if removed:
        print(f"  (pruned {removed} old backups for {path.stem}; keeping newest {MAX_BACKUP_KEEP})", flush=True)


def _save_checkpoint_atomic(obj, path):
    """Use atomic_torch_save from cell 4 if available, else direct torch.save."""
    try:
        return atomic_torch_save(obj, path)
    except NameError:
        torch.save(obj, path)
        return path


def _sanity_check_first_batch(loader, *, n_classes: int, label: str = "ld_tr") -> None:
    """Pull one batch and assert shapes/labels are sane before launching long training.
    Catches teacher-dim drift and out-of-range labels (the common CUDA device-side assert)."""
    try:
        crops_b, t, y = next(iter(loader))
    except Exception as err:
        print(f"  sanity-check ({label}) skipped: {err}")
        return
    if t.shape[-1] != int(_teach_dim):
        raise ValueError(
            f"sanity-check ({label}): teacher dim {t.shape[-1]} != expected {_teach_dim}"
        )
    n_views_seen = len(crops_b[0]) if crops_b else 0
    n_views_expect = 1 + N_NATIVE_RANDOM
    if n_views_seen and n_views_seen != n_views_expect:
        print(f"  sanity-check ({label}): views/sample={n_views_seen} (expected {n_views_expect})")
    if y.numel():
        t_min = int(y.min().item())
        t_max = int(y.max().item())
        if t_max >= n_classes:
            raise ValueError(
                f"sanity-check ({label}): max target {t_max} >= num_classes {n_classes}; "
                "would trigger CUDA device-side assert in CrossEntropyLoss. "
                "Update NUM_MATERIAL_CLASSES or remap labels."
            )
        if t_min < -1:
            raise ValueError(
                f"sanity-check ({label}): target {t_min} < -1 (-1 reserved for unlabeled)."
            )
    else:
        t_min = t_max = None
    print(
        f"  sanity-check ({label}) ok | batch={len(crops_b)} | views/sample={n_views_seen} "
        f"| teacher={tuple(t.shape)} | targets in [{t_min}, {t_max}] / K={n_classes}"
    )


def _milestone_epochs(n_epochs: int):
    return sorted(
        {
            max(0, min(int(n_epochs) - 1, int(round(int(n_epochs) * float(f)))))
            for f in (0.25, 0.6)
        }
    )


if not RUN_PROGRESSIVE_VIT_FINETUNE:
    print("Set RUN_PROGRESSIVE_VIT_FINETUNE = True in the config cell to run training.")
else:
    FINETUNE_CKPT_DIR.mkdir(parents=True, exist_ok=True)
    for _param in student.parameters():
        _param.requires_grad = True
    _run_t0 = _time.time()
    _stage_summaries: list[dict] = []
    print("--- pre-flight sanity check on first val batch (catches dim/label issues early) ---")
    _sanity_check_first_batch(ld_va, n_classes=NUM_MATERIAL_CLASSES, label="ld_va")

    vit_schedule = [
        ("A_readout", lambda: unfreeze_clip_visual_ln_post_proj_only(visual), EPOCHS_VIT_A, LR_STUDENT_A, LR_READOUT_A, None, False),
        ("B_last2", lambda: unfreeze_clip_visual_last_n_blocks(visual, 2), EPOCHS_VIT_B, LR_STUDENT_B, LR_READOUT_B, LR_BACKBONE_B, False),
        ("C_last4", lambda: unfreeze_clip_visual_last_n_blocks(visual, 4), EPOCHS_VIT_C4, LR_STUDENT_C, LR_READOUT_C, LR_BACKBONE_C4, False),
        ("C_last8", lambda: unfreeze_clip_visual_last_n_blocks(visual, 8), EPOCHS_VIT_C8, LR_STUDENT_C, LR_READOUT_C, LR_BACKBONE_C8, False),
        ("C_last12", lambda: unfreeze_clip_visual_last_n_blocks(visual, 12), EPOCHS_VIT_C12, LR_STUDENT_C, LR_READOUT_C, LR_BACKBONE_C12, False),
        ("D_full_decay", lambda: unfreeze_clip_visual_full(visual), EPOCHS_VIT_D, LR_STUDENT_D, LR_READOUT_D, LR_BACKBONE_LAST_D, True),
    ]

    vit_schedule = [st for st in vit_schedule if st[2] > 0]

    run_z = bool(RUN_EMBEDDING_STUDENT_WARMUP)
    if SCHEDULE_START_STAGE is not None:
        if SCHEDULE_START_STAGE == "Z_embed":
            if not run_z:
                raise ValueError("SCHEDULE_START_STAGE='Z_embed' requires RUN_EMBEDDING_STUDENT_WARMUP=True")
            vit_schedule = []
            print("SCHEDULE_START_STAGE='Z_embed' → ViT stages skipped (embedding warmup only).", flush=True)
        else:
            names = [s[0] for s in vit_schedule]
            if SCHEDULE_START_STAGE not in names:
                raise ValueError(
                    f"SCHEDULE_START_STAGE={SCHEDULE_START_STAGE!r} not in {['Z_embed'] + names!r}"
                )
            vit_schedule = vit_schedule[names.index(SCHEDULE_START_STAGE) :]
            print(f"SCHEDULE_START_STAGE={SCHEDULE_START_STAGE!r} → {[s[0] for s in vit_schedule]}", flush=True)
            run_z = False

    stage_list_log = (["Z_embed"] if run_z else []) + [s[0] for s in vit_schedule]
    print(
        f"Training stages: {stage_list_log} | "
        f"ViT batches/epoch (when active): {len(ld_tr)} train, {len(ld_va)} val | "
        f"batch_size A={BATCH_SIZE} B<={BATCH_SIZE_VIT_SUBSEQUENT_MAX} "
        f"C<={BATCH_SIZE_STAGE_C} D<={BATCH_SIZE_STAGE_D}",
        flush=True,
    )

    _append_training_epoch_log(
        {
            "event": "run_start",
            "schedule": stage_list_log,
            "clip_model": CLIP_MODEL,
            "batch_size_default": BATCH_SIZE,
            "batch_size_stage_c": BATCH_SIZE_STAGE_C,
            "stage_c_fp32": STAGE_C_USE_FP32,
            "stage_d_fp32": STAGE_D_USE_FP32,
            "stage_c_encode_chunk": STAGE_C_VISUAL_ENCODE_CHUNK_SIZE,
            "use_mixed_precision": USE_MIXED_PRECISION,
            "use_ce_class_weights": USE_CE_CLASS_WEIGHTS,
            "dataloader_num_workers": DATALOADER_NUM_WORKERS,
            "colab_pro_tuning": bool(_ON_COLAB),
            "run_embedding_student_warmup": run_z,
            "best_checkpoint_metric": "val_acc_labeled",
        }
    )

    best_val_acc = -1.0
    best_bundle = None

    def _snapshot_finetune_bundle(model, **kwargs):
        """Detach state_dict so best_bundle does not alias live parameters (in-place training)."""
        d = unified_finetune_checkpoint_dict(model, **kwargs)
        d["state_dict"] = copy.deepcopy(d["state_dict"])
        return d

    last_tr = last_va = None
    _active_batch_size = None

    if run_z:
        freeze_clip_visual(visual)
        visual.eval()
        _ebs = EMBED_WARMUP_BATCH_SIZE
        if _ebs is None or int(_ebs) <= 0:
            _bs_emb = max(len(train_ix), 1)
        else:
            _bs_emb = min(int(_ebs), max(len(train_ix), 1))
        _bs_va = max(len(val_ix), 1)
        _ds_z_tr_base = sc.PLMMaterialDistillDataset(train_ix, z_embed_image_input, fused, material_targets)
        ds_z_va = sc.PLMMaterialDistillDataset(val_ix, z_embed_image_input, fused, material_targets)
        ds_z_tr = _TeacherNoiseDatasetWrapper(
            _ds_z_tr_base,
            noise_std=TEACHER_TARGET_NOISE_STD,
            t_scale=fused[train_ix].std(axis=0).astype(np.float32) if TEACHER_TARGET_NOISE_STD > 0 else None,
        )
        # Z_embed sampler: same WeightedRandomSampler logic. Z trains on tensors (cheap),
        # so per-row weights are essentially free here too.
        z_sampler = None
        if USE_CLASS_BALANCED_SAMPLER and SAMPLER_WEIGHT_KIND != "none":
            z_sampler, _ = build_class_balanced_sampler(y_train_arr, kind=SAMPLER_WEIGHT_KIND)
        ld_z_tr = DataLoader(
            ds_z_tr,
            batch_size=_bs_emb,
            shuffle=(z_sampler is None),
            sampler=z_sampler,
            num_workers=int(DATALOADER_NUM_WORKERS),
            pin_memory=dataloader_pin_memory(torch_device),
        )
        ld_z_va = DataLoader(
            ds_z_va,
            batch_size=_bs_va,
            shuffle=False,
            num_workers=int(DATALOADER_NUM_WORKERS),
            pin_memory=dataloader_pin_memory(torch_device),
        )
        opt_z = torch.optim.AdamW(
            student.parameters(),
            lr=LR_EMBED_WARMUP,
            weight_decay=EMBED_WARMUP_WEIGHT_DECAY,
        )
        sched_z = torch.optim.lr_scheduler.MultiStepLR(
            opt_z, milestones=_milestone_epochs(EPOCHS_EMBED_WARMUP), gamma=0.5
        )
        print(
            f"=== Z_embed | student on LongCLIP-B image embeddings ({_IMG_D}-D) -> teacher {edim}-D | "
            f"trainable student only | batches/epoch {len(ld_z_tr)}/{len(ld_z_va)} train_batch={_bs_emb} val_batch={_bs_va}",
            flush=True,
        )
        best_z_acc = -1.0
        best_z_state = None
        stall_z = 0
        for epoch in range(EPOCHS_EMBED_WARMUP):
            tr = sc.student_embedding_joint_epoch(
                student,
                ld_z_tr,
                opt_z,
                torch_device,
                train=True,
                lambda_emb=LAMBDA_EMB_P3,
                lambda_ce=LAMBDA_CE_P3,
                emb_kind=EMB_LOSS_KIND,
                label_smoothing=LABEL_SMOOTHING_P3,
                ce_class_weights=None,
                grad_clip=EMBED_WARMUP_GRAD_CLIP,
            )
            va = sc.student_embedding_joint_epoch(
                student,
                ld_z_va,
                None,
                torch_device,
                train=False,
                lambda_emb=LAMBDA_EMB_P3,
                lambda_ce=LAMBDA_CE_P3,
                emb_kind=EMB_LOSS_KIND,
                label_smoothing=LABEL_SMOOTHING_P3,
                ce_class_weights=None,
                grad_clip=0.0,
            )
            last_tr, last_va = tr, va
            vaz = float(va[3])
            improved_z = not math.isnan(vaz) and vaz > best_z_acc + 1e-8
            is_new_best_val = improved_z
            print(
                f"Z_embed epoch {epoch + 1}/{EPOCHS_EMBED_WARMUP}  "
                f"train_loss={tr[0]:.5f}  val_loss={va[0]:.5f}  "
                f"train_acc_labeled={tr[3]:.4f}  val_acc_labeled={va[3]:.4f}",
                flush=True,
            )
            if improved_z:
                best_z_acc = float(vaz)
                best_z_state = copy.deepcopy(student.state_dict())
                stall_z = 0
            elif EARLY_STOP_PATIENCE_EMBED is not None:
                stall_z += 1
            _append_training_epoch_log(
                {
                    "event": "epoch_end",
                    "stage": "Z_embed",
                    "epoch_in_stage": epoch + 1,
                    "epochs_in_stage": EPOCHS_EMBED_WARMUP,
                    "train_loss": tr[0],
                    "val_loss": va[0],
                    "train_emb": tr[1],
                    "val_emb": va[1],
                    "train_ce": tr[2],
                    "val_ce": va[2],
                    "train_acc_labeled": tr[3],
                    "val_acc_labeled": va[3],
                    "best_val_acc_so_far": best_z_acc,
                    "batch_size": _bs_emb,
                    "val_batch_size": _bs_va,
                    "embed_stall_epochs": stall_z,
                    "early_stop_patience_embed": EARLY_STOP_PATIENCE_EMBED,
                    "is_new_best": is_new_best_val,
                }
            )
            if (
                EARLY_STOP_PATIENCE_EMBED is not None
                and stall_z >= EARLY_STOP_PATIENCE_EMBED
            ):
                print(
                    f"Z_embed early stop: no val_acc_labeled improvement for {EARLY_STOP_PATIENCE_EMBED} epochs "
                    f"(best={best_z_acc:.4f})",
                    flush=True,
                )
                sched_z.step()
                break
            sched_z.step()
        if best_z_state is not None:
            student.load_state_dict(best_z_state)
            print(f"Z_embed restored best val_acc_labeled={best_z_acc:.4f}", flush=True)
        z_ckpt = FINETUNE_CKPT_DIR / "vit_finetune_stage_B_Z_embed_student_head.pth"
        backup_checkpoint_if_exists(z_ckpt)
        _save_checkpoint_atomic(
            unified_finetune_checkpoint_dict(
                full_model,
                meta={
                    **dict(meta),
                    "finetune_clip_model": CLIP_MODEL,
                    "num_views": 1 + N_NATIVE_RANDOM,
                    "window_sample_seed": WINDOW_SAMPLE_SEED,
                    "epoch_training_log_jsonl": str(TRAINING_LOG_JSONL),
                    "best_checkpoint_metric": "val_acc_labeled",
                },
                stage="Z_embed",
                epoch_in_stage=EPOCHS_EMBED_WARMUP - 1,
                val_loss=float(last_va[0]),
                val_acc=float(last_va[3]),
            ),
            z_ckpt,
        )
        print(f"  checkpoint: {z_ckpt.name}", flush=True)
        _stage_summaries.append({
            "stage": "Z_embed",
            "best_val_acc": float(best_z_acc) if best_z_acc >= 0 else None,
            "epochs_run": int(epoch + 1),
            "max_epochs": int(EPOCHS_EMBED_WARMUP),
            "checkpoint": str(z_ckpt),
        })

    z_ref_val = float(best_z_acc) if run_z else None
    if run_z:
        best_val_acc = -1.0
        best_bundle = None

    for stage_name, apply_unfreeze, n_ep, lr_st, lr_ro, lr_bb, full_decay in vit_schedule:
        apply_unfreeze()
        student.set_head_trainable(True)
        student.set_classifier_trainable(True)
        if stage_name == "A_readout":
            desired_bs = int(BATCH_SIZE)
        elif stage_name.startswith("C_"):
            desired_bs = min(
                int(BATCH_SIZE),
                int(BATCH_SIZE_VIT_SUBSEQUENT_MAX),
                int(BATCH_SIZE_STAGE_C),
            )
        elif stage_name == "D_full_decay":
            desired_bs = min(
                int(BATCH_SIZE),
                int(BATCH_SIZE_VIT_SUBSEQUENT_MAX),
                int(BATCH_SIZE_STAGE_D),
            )
        else:
            desired_bs = min(int(BATCH_SIZE), int(BATCH_SIZE_VIT_SUBSEQUENT_MAX))
        if _active_batch_size != desired_bs:
            _stage_sampler = None
            if USE_CLASS_BALANCED_SAMPLER and SAMPLER_WEIGHT_KIND != "none":
                _stage_sampler, _ = build_class_balanced_sampler(y_train_arr, kind=SAMPLER_WEIGHT_KIND)
            ld_tr = make_vit_dataloader_v2(ds_tr, batch_size=desired_bs, shuffle=True, sampler=_stage_sampler)
            ld_va = make_vit_dataloader_v2(ds_va, batch_size=desired_bs, shuffle=False, sampler=None)
            _active_batch_size = desired_bs
            print(
                f"  DataLoader batch_size={desired_bs}  "
                f"(train {len(ld_tr)} batches / epoch, val {len(ld_va)})",
                flush=True,
            )

        encode_chunk = STAGE_C_VISUAL_ENCODE_CHUNK_SIZE if stage_name.startswith("C_") else None
        fp32_here = (stage_name.startswith("C_") and STAGE_C_USE_FP32) or (
            stage_name == "D_full_decay" and STAGE_D_USE_FP32
        )

        n_tr_v = sum(p.numel() for p in visual.parameters() if p.requires_grad)
        n_tr_s = sum(p.numel() for p in student.parameters() if p.requires_grad)
        print(f"=== {stage_name} | trainable: visual {n_tr_v:,} + student {n_tr_s:,}")
        if fp32_here or encode_chunk is not None:
            print(
                f"  precision: fp32_no_amp={fp32_here}  visual_encode_chunk_size={encode_chunk}",
                flush=True,
            )

        opt = build_adamw_vit_finetune(
            visual,
            student,
            lr_student=lr_st,
            lr_readout=lr_ro,
            lr_backbone=lr_bb,
            weight_decay=WEIGHT_DECAY_FINETUNE,
            full_visual_layer_decay=full_decay,
            layer_decay=LAYER_DECAY_FULL,
        )

        _is_a_stage = stage_name == "A_readout"
        if _is_a_stage:
            _ep_limit = int(A_VIT_MAX_EPOCHS_PER_STAGE)
            _a_min_ep = max(1, int(A_VIT_MIN_EPOCHS_PER_STAGE))
            _a_pat = A_VIT_EARLY_STOP_PATIENCE
            _a_best_stage = -1.0
            _a_stall = 0
            # Acceptance gate: only refuse A bundles if explicitly enabled. Z_embed has the
            # unfair advantage of frozen features, so by default we let A compete fairly.
            _min_acc_a = float(z_ref_val) if (z_ref_val is not None and bool(A_VIT_PROMOTE_MIN_VS_Z)) else -1.0
        else:
            _ep_limit = int(n_ep)
            _a_min_ep = 1
            _a_pat = EARLY_STOP_PATIENCE_VIT.get(stage_name)
            _a_best_stage = -1.0
            _a_stall = 0
            _min_acc_a = -1.0

        stage_best_val = float("-inf")
        stage_best_bundle = None
        _stage_t0 = _time.time()

        _swa_active = bool(USE_SWA_IN_STAGE_D and stage_name == "D_full_decay" and _ep_limit > 0)
        _swa_model = None
        _swa_count = 0
        if _swa_active:
            try:
                from torch.optim.swa_utils import AveragedModel
                _swa_model = AveragedModel(full_model)
                print(
                    f"  SWA: armed for last {min(int(SWA_LAST_N_EPOCHS), _ep_limit)} epoch(s) of {stage_name}",
                    flush=True,
                )
            except Exception as _swa_err:
                print(f"  SWA init failed ({_swa_err}); proceeding without averaging.", flush=True)
                _swa_active = False
                _swa_model = None

        for epoch in range(_ep_limit):
            tr = vit_finetune_run_epoch(
                visual,
                student,
                preprocess,
                ld_tr,
                opt,
                torch_device,
                train=True,
                lambda_emb=LAMBDA_EMB_P3,
                lambda_ce=LAMBDA_CE_P3,
                emb_kind=EMB_LOSS_KIND,
                label_smoothing=LABEL_SMOOTHING_P3,
                grad_clip=GRAD_CLIP_FINETUNE,
                use_tqdm=SHOW_TQDM_BATCHES,
                tqdm_desc=f"{stage_name} train {epoch + 1}/{_ep_limit}",
                log_every_batches=LOG_EVERY_N_TRAIN_BATCHES,
                log_prefix=f"{stage_name} train",
                student_frozen_eval=False,
                ce_class_weights=None,
                visual_encode_chunk_size=encode_chunk,
                visual_ln_post_only=VIT_STAGE_USE_LN_POST_ENCODING,
                **_vit_finetune_amp_kwargs(for_training=True, stage_name=stage_name),
            )
            va = vit_finetune_run_epoch(
                visual,
                student,
                preprocess,
                ld_va,
                None,
                torch_device,
                train=False,
                lambda_emb=LAMBDA_EMB_P3,
                lambda_ce=LAMBDA_CE_P3,
                emb_kind=EMB_LOSS_KIND,
                label_smoothing=LABEL_SMOOTHING_P3,
                grad_clip=0.0,
                use_tqdm=SHOW_TQDM_BATCHES,
                tqdm_desc=f"{stage_name} val {epoch + 1}/{_ep_limit}",
                log_every_batches=LOG_EVERY_N_VAL_BATCHES,
                log_prefix=f"{stage_name} val",
                student_frozen_eval=False,
                ce_class_weights=None,
                visual_encode_chunk_size=encode_chunk,
                visual_ln_post_only=VIT_STAGE_USE_LN_POST_ENCODING,
                **_vit_finetune_amp_kwargs(for_training=False, stage_name=stage_name),
            )
            last_tr, last_va = tr, va
            print(
                f"{stage_name} epoch {epoch + 1}/{_ep_limit}  "
                f"train_loss={tr[0]:.5f}  val_loss={va[0]:.5f}  "
                f"train_acc_labeled={tr[3]:.4f}  val_acc_labeled={va[3]:.4f}",
                flush=True,
            )
            _va3 = float(va[3])
            if not math.isnan(_va3) and _va3 > stage_best_val + 1e-8:
                stage_best_val = float(_va3)
                stage_best_bundle = _snapshot_finetune_bundle(
                    full_model,
                    meta={
                        **dict(meta),
                        "finetune_clip_model": CLIP_MODEL,
                        "schedule": stage_list_log,
                        "emb_loss_kind": EMB_LOSS_KIND,
                        "lambda_emb": LAMBDA_EMB_P3,
                        "lambda_ce": LAMBDA_CE_P3,
                        "label_smoothing": LABEL_SMOOTHING_P3,
                        "num_views": 1 + N_NATIVE_RANDOM,
                        "window_sample_seed": WINDOW_SAMPLE_SEED,
                        "geom_augment_train": GEOM_AUGMENT_TRAIN,
                        "best_checkpoint_metric": "val_acc_labeled",
                    },
                    stage=stage_name,
                    epoch_in_stage=epoch,
                    val_loss=float(va[0]),
                    val_acc=float(_va3),
                )
            _improved_global = not math.isnan(_va3) and _va3 > best_val_acc + 1e-8
            _accept_snapshot = (not _is_a_stage) or (_min_acc_a < 0) or (
                _va3 >= _min_acc_a - 1e-8
            )
            _amp_tr = _vit_finetune_amp_kwargs(for_training=True, stage_name=stage_name)
            _append_training_epoch_log(
                {
                    "event": "epoch_end",
                    "stage": stage_name,
                    "epoch_in_stage": epoch + 1,
                    "epochs_in_stage": _ep_limit,
                    "train_loss": tr[0],
                    "val_loss": va[0],
                    "train_emb": tr[1],
                    "val_emb": va[1],
                    "train_ce": tr[2],
                    "val_ce": va[2],
                    "train_acc_labeled": tr[3],
                    "val_acc_labeled": va[3],
                    "best_val_acc_so_far": best_val_acc,
                    "batch_size": _active_batch_size,
                    "use_amp": bool(_amp_tr.get("use_amp")),
                    "fp32_no_amp": fp32_here,
                    "visual_encode_chunk_size": encode_chunk,
                    "is_new_best": bool(_improved_global and _accept_snapshot),
                    "a_vit_min_acc_z_embed": _min_acc_a if _is_a_stage else None,
                    "a_vit_stall_epochs": _a_stall if _is_a_stage else None,
                }
            )
            if _improved_global and _accept_snapshot:
                best_val_acc = float(_va3)
                best_bundle = _snapshot_finetune_bundle(
                    full_model,
                    meta={
                        **dict(meta),
                        "finetune_clip_model": CLIP_MODEL,
                        "schedule": stage_list_log,
                        "emb_loss_kind": EMB_LOSS_KIND,
                        "lambda_emb": LAMBDA_EMB_P3,
                        "lambda_ce": LAMBDA_CE_P3,
                        "label_smoothing": LABEL_SMOOTHING_P3,
                        "num_views": 1 + N_NATIVE_RANDOM,
                        "window_sample_seed": WINDOW_SAMPLE_SEED,
                        "geom_augment_train": GEOM_AUGMENT_TRAIN,
                        "best_checkpoint_metric": "val_acc_labeled",
                    },
                    stage=stage_name,
                    epoch_in_stage=epoch,
                    val_loss=float(va[0]),
                    val_acc=float(_va3),
                )
            if not math.isnan(_va3) and _va3 > _a_best_stage + 1e-8:
                _a_best_stage = float(_va3)
                _a_stall = 0
            elif _a_pat is not None:
                _a_stall += 1
            if (
                _swa_active
                and _swa_model is not None
                and (epoch + 1) > (_ep_limit - int(SWA_LAST_N_EPOCHS))
            ):
                _swa_model.update_parameters(full_model)
                _swa_count += 1
                print(f"  SWA: averaged in epoch {epoch + 1}/{_ep_limit} (n_avg={_swa_count})", flush=True)

            if (
                _a_pat is not None
                and epoch + 1 >= _a_min_ep
                and _a_stall >= int(_a_pat)
            ):
                print(
                    f"{stage_name}: early stop (no val_acc_labeled gain for {_a_pat} epochs; "
                    f"best in stage={_a_best_stage:.4f})",
                    flush=True,
                )
                break



        if _swa_active and _swa_model is not None and _swa_count > 0:
            try:
                _pre_swa_state = copy.deepcopy(full_model.state_dict())
                full_model.load_state_dict(_swa_model.module.state_dict(), strict=True)
                print(f"  SWA: evaluating averaged weights ({_swa_count} snapshots) on val...", flush=True)
                _swa_va = vit_finetune_run_epoch(
                    visual,
                    student,
                    preprocess,
                    ld_va,
                    None,
                    torch_device,
                    train=False,
                    lambda_emb=LAMBDA_EMB_P3,
                    lambda_ce=LAMBDA_CE_P3,
                    emb_kind=EMB_LOSS_KIND,
                    label_smoothing=LABEL_SMOOTHING_P3,
                    grad_clip=0.0,
                    use_tqdm=SHOW_TQDM_BATCHES,
                    tqdm_desc=f"{stage_name} SWA val",
                    log_every_batches=LOG_EVERY_N_VAL_BATCHES,
                    log_prefix=f"{stage_name} SWA val",
                    student_frozen_eval=False,
                    ce_class_weights=None,
                    visual_encode_chunk_size=encode_chunk,
                    visual_ln_post_only=VIT_STAGE_USE_LN_POST_ENCODING,
                    **_vit_finetune_amp_kwargs(for_training=False, stage_name=stage_name),
                )
                _swa_va3 = float(_swa_va[3])
                print(
                    f"  SWA val_acc_labeled={_swa_va3:.4f}  val_loss={_swa_va[0]:.5f}  "
                    f"(stage_best_val={stage_best_val:.4f}, global_best_val_acc={best_val_acc:.4f})",
                    flush=True,
                )
                _swa_meta = {
                    **dict(meta),
                    "finetune_clip_model": CLIP_MODEL,
                    "schedule": stage_list_log,
                    "emb_loss_kind": EMB_LOSS_KIND,
                    "lambda_emb": LAMBDA_EMB_P3,
                    "lambda_ce": LAMBDA_CE_P3,
                    "label_smoothing": LABEL_SMOOTHING_P3,
                    "num_views": 1 + N_NATIVE_RANDOM,
                    "window_sample_seed": WINDOW_SAMPLE_SEED,
                    "geom_augment_train": GEOM_AUGMENT_TRAIN,
                    "best_checkpoint_metric": "val_acc_labeled",
                    "is_swa_average": True,
                    "swa_count": int(_swa_count),
                }
                if not math.isnan(_swa_va3) and _swa_va3 > stage_best_val + 1e-8:
                    stage_best_val = _swa_va3
                    stage_best_bundle = _snapshot_finetune_bundle(
                        full_model,
                        meta=_swa_meta,
                        stage=stage_name,
                        epoch_in_stage=epoch,
                        val_loss=float(_swa_va[0]),
                        val_acc=_swa_va3,
                    )
                    print("  SWA beats stage-best; using SWA weights for stage checkpoint.", flush=True)
                if not math.isnan(_swa_va3) and _swa_va3 > best_val_acc + 1e-8:
                    best_val_acc = _swa_va3
                    best_bundle = _snapshot_finetune_bundle(
                        full_model,
                        meta=_swa_meta,
                        stage=stage_name,
                        epoch_in_stage=epoch,
                        val_loss=float(_swa_va[0]),
                        val_acc=_swa_va3,
                    )
                    print("  SWA beats GLOBAL best; promoted to global-best bundle.", flush=True)
                else:
                    full_model.load_state_dict(_pre_swa_state, strict=True)
            except Exception as _swa_err:
                print(f"  SWA finalize failed ({_swa_err}); keeping pre-SWA weights.", flush=True)
                try:
                    full_model.load_state_dict(_pre_swa_state, strict=True)
                except Exception:
                    pass

        if stage_best_bundle is not None:
            full_model.load_state_dict(stage_best_bundle["state_dict"])
            _save_va = float(stage_best_val)
            _save_vl = float(stage_best_bundle.get("val_loss", last_va[0]))
            print(
                f"{stage_name}: restored stage-best weights for stage checkpoint "
                f"(val_acc_labeled={_save_va:.4f})",
                flush=True,
            )
        else:
            _save_va = float(last_va[3])
            _save_vl = float(last_va[0])
        stage_ckpt = FINETUNE_CKPT_DIR / f"vit_finetune_stage_B_{stage_name}.pth"
        backup_checkpoint_if_exists(stage_ckpt)
        _save_checkpoint_atomic(
            unified_finetune_checkpoint_dict(
                full_model,
                meta={
                    **dict(meta),
                    "finetune_clip_model": CLIP_MODEL,
                    "num_views": 1 + N_NATIVE_RANDOM,
                    "window_sample_seed": WINDOW_SAMPLE_SEED,
                    "epoch_training_log_jsonl": str(TRAINING_LOG_JSONL),
                    "best_checkpoint_metric": "val_acc_labeled",
                },
                stage=stage_name,
                epoch_in_stage=epoch,
                val_loss=_save_vl,
                val_acc=_save_va,
            ),
            stage_ckpt,
        )
        print(f"  checkpoint: {stage_ckpt.name}")
        _stage_summaries.append({
            "stage": stage_name,
            "best_val_acc": float(stage_best_val) if stage_best_val > float("-inf") else None,
            "epochs_run": int(epoch + 1),
            "max_epochs": int(_ep_limit),
            "wall_seconds": round(_time.time() - _stage_t0, 1),
            "batch_size": int(_active_batch_size) if _active_batch_size else None,
            "checkpoint": str(stage_ckpt),
        })

    if best_bundle is not None:
        backup_checkpoint_if_exists(FINETUNE_CKPT_BEST)
        _save_checkpoint_atomic(best_bundle, FINETUNE_CKPT_BEST)
        print(f"Best val_acc_labeled {best_val_acc:.5f} → {FINETUNE_CKPT_BEST}")
        print(
            "Deploy: ClipVisualStudent + load_clip_visual_student_checkpoint; "
            f"forward with num_views={1 + N_NATIVE_RANDOM}."
        )

    _run_secs = _time.time() - _run_t0
    print("\n" + "=" * 72)
    print(f"Run summary | total wall time: {_run_secs / 60:.1f} min ({_run_secs:.0f}s)")
    print("=" * 72)
    for s in _stage_summaries:
        bv = "  -  " if s.get("best_val_acc") is None else f"{s['best_val_acc']:.4f}"
        ws = s.get("wall_seconds")
        ws_s = f" wall={ws}s" if ws else ""
        print(
            f"  {s['stage']:<14} best_val_acc={bv}  epochs={s.get('epochs_run')}/{s.get('max_epochs')}{ws_s}"
        )
    if best_bundle is not None:
        print(f"  GLOBAL BEST    val_acc_labeled={best_val_acc:.4f}  → {FINETUNE_CKPT_BEST.name}")
    summary_path = FINETUNE_CKPT_DIR / "vit_finetune_run_summary.json"
    try:
        with open(summary_path, "w", encoding="utf-8") as f:
            _json.dump(
                {
                    "ts_utc": datetime.now(timezone.utc).isoformat(),
                    "wall_seconds": _run_secs,
                    "best_val_acc": float(best_val_acc) if best_val_acc > -1 else None,
                    "best_checkpoint": str(FINETUNE_CKPT_BEST) if best_bundle is not None else None,
                    "stages": _stage_summaries,
                    "schedule": stage_list_log,
                    "num_material_classes": int(NUM_MATERIAL_CLASSES),
                    "image_dim": int(_IMG_D),
                    "teacher_dim": int(edim),
                },
                f,
                indent=2,
            )
        print(f"  summary json:  {summary_path}")
    except Exception as _sum_err:
        print(f"  (run summary write failed: {_sum_err})")
